<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/13_federated_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent storage
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Configure Git remote access
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "your_email@example.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Synchronize repo
%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

# 4. Inject module search path
src_str = str(SRC_DIR.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)

# 5. Install dependencies
!pip install -q transformers pyarrow scikit-learn

print("Environment setup and path injection complete.")

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 154 (delta 65), reused 99 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 518.80 KiB | 10.17 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/DP-MMFL
Environment setup and path injection complete.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"
E2_DIR = PROJECT_ROOT / "results" / "E2_multimodal"
E3_DIR = PROJECT_ROOT / "results" / "E3_federated"

CHECKPOINT_DIR = E3_DIR / "checkpoints"
METRICS_DIR = E3_DIR / "metrics"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load exact split from E2
with open(E2_DIR / "split_info.json", "r") as f:
    split_info = json.load(f)

train_sample_ids = [int(x) for x in split_info["train_sample_ids"]]
val_sample_ids = [int(x) for x in split_info["val_sample_ids"]]

# 2. Extract split rows
manifest = pd.read_parquet(MANIFEST_PATH)
train_rows = manifest[
    manifest["sample_id"].isin(train_sample_ids)
    & (manifest["frontal_lateral"] == "Frontal")
].copy()
val_rows = manifest[
    manifest["sample_id"].isin(val_sample_ids)
    & (manifest["frontal_lateral"] == "Frontal")
].copy()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("--- Split Integrity Profile ---")
print("Device:                         ", device)
print("Training images:                ", len(train_rows))
print("Validation images:              ", len(val_rows))
print("Train patients:                 ", train_rows["deid_patient_id"].nunique())
print("Validation patients:            ", val_rows["deid_patient_id"].nunique())
print(
    "Train/validation patient overlap:",
    len(set(train_rows["deid_patient_id"]) & set(val_rows["deid_patient_id"])),
)

# 3. Assert exact data policy guarantees
assert len(train_rows) == 835, f"Expected 835 train images, found {len(train_rows)}"
assert len(val_rows) == 165, f"Expected 165 val images, found {len(val_rows)}"
assert set(train_rows["deid_patient_id"]).isdisjoint(
    set(val_rows["deid_patient_id"])
), "Data leakage between train and validation splits!"

print("\nSplit assertions: ALL CHECKS PASSED")

--- Split Integrity Profile ---
Device:                          cuda:0
Training images:                 835
Validation images:               165
Train patients:                  269
Validation patients:             68
Train/validation patient overlap: 0

Split assertions: ALL CHECKS PASSED


In [ ]:
import numpy as np

NUM_CLIENTS = 3
SEED = 42

patient_ids = train_rows["deid_patient_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(SEED)
rng.shuffle(patient_ids)
patient_partitions = np.array_split(patient_ids, NUM_CLIENTS)

client_sample_ids = {}

print("--- Client Partition Summary ---")
for client_idx, client_patients in enumerate(patient_partitions):
    client_id = f"client_{client_idx + 1}"
    client_rows = train_rows[
        train_rows["deid_patient_id"].isin(client_patients)
    ]
    client_sample_ids[client_id] = (
        client_rows["sample_id"].astype(int).tolist()
    )
    print(
        f"{client_id}: "
        f"{len(client_rows)} images, "
        f"{len(client_patients)} patients"
    )

# 1. Verify all training samples are uniquely accounted for
assigned_ids = [
    sample_id
    for ids in client_sample_ids.values()
    for sample_id in ids
]
assert len(assigned_ids) == len(train_sample_ids), "Assigned count mismatch!"
assert len(set(assigned_ids)) == len(train_sample_ids), "Duplicate samples found across clients!"
assert set(assigned_ids) == set(train_sample_ids), "Assigned set does not match training set!"

# 2. Verify strict patient isolation across all client pairs
for i in range(NUM_CLIENTS):
    for j in range(i + 1, NUM_CLIENTS):
        assert set(patient_partitions[i]).isdisjoint(
            set(patient_partitions[j])
        ), f"Patient overlap found between client_{i+1} and client_{j+1}!"

print("\nClient partition validation: PASS")

--- Client Partition Summary ---
client_1: 257 images, 90 patients
client_2: 314 images, 90 patients
client_3: 264 images, 89 patients

Client partition validation: PASS


In [ ]:
partition_path = METRICS_DIR / "client_partitions.json"

with open(partition_path, "w") as f:
    json.dump(
        {
            "num_clients": NUM_CLIENTS,
            "seed": SEED,
            "client_sample_ids": client_sample_ids,
        },
        f,
        indent=2,
    )

print("Saved client partitions:", partition_path)
print("File exists:            ", partition_path.exists())

Saved client partitions: /content/drive/MyDrive/DP-MMFL/results/E3_federated/metrics/client_partitions.json
File exists:             True


In [ ]:
from pathlib import Path
import torch

E2_DIR = PROJECT_ROOT / "results/E2_multimodal"

checkpoint_candidates = [
    E2_DIR / "checkpoints/best.pt",
    E2_DIR / "checkpoints/last.pt",
]

embedding_path = E2_DIR / "text_embeddings.pt"

print("Checkpoint candidates:")
for path in checkpoint_candidates:
    print(f"  {path.name}: {path.exists()}")

print("\nText embeddings:")
print(f"  Exists: {embedding_path.exists()}")

if embedding_path.exists():
    embedding_data = torch.load(
        embedding_path,
        map_location="cpu",
        weights_only=False
    )
    print("  Embedding shape:", embedding_data["embeddings"].shape)
    print("  Number of sample IDs:", len(embedding_data["sample_ids"]))
    print("  Unique sample IDs:",
          len(set(map(int, embedding_data["sample_ids"]))))

Checkpoint candidates:
  best.pt: True
  last.pt: True

Text embeddings:
  Exists: True
  Embedding shape: torch.Size([1000, 768])
  Number of sample IDs: 1000
  Unique sample IDs: 1000


In [ ]:

import torch

checkpoint_path = E2_DIR / "checkpoints/best.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
        print("State dict key: model_state_dict")
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
        print("State dict key: state_dict")
    elif checkpoint and all(
        isinstance(v, torch.Tensor) for v in checkpoint.values()
    ):
        state_dict = checkpoint
        print("Checkpoint is a raw state dict")
    else:
        state_dict = None
        print("No standard model state-dict key detected")

    if state_dict is not None:
        print("Number of parameter entries:", len(state_dict))
        print("First 12 parameter names:")
        for key in list(state_dict.keys())[:12]:
            print(" ", key)


Checkpoint type: OrderedDict
Top-level keys: ['image_encoder.conv1.weight', 'image_encoder.bn1.weight', 'image_encoder.bn1.bias', 'image_encoder.bn1.running_mean', 'image_encoder.bn1.running_var', 'image_encoder.bn1.num_batches_tracked', 'image_encoder.layer1.0.conv1.weight', 'image_encoder.layer1.0.bn1.weight', 'image_encoder.layer1.0.bn1.bias', 'image_encoder.layer1.0.bn1.running_mean', 'image_encoder.layer1.0.bn1.running_var', 'image_encoder.layer1.0.bn1.num_batches_tracked', 'image_encoder.layer1.0.conv2.weight', 'image_encoder.layer1.0.bn2.weight', 'image_encoder.layer1.0.bn2.bias', 'image_encoder.layer1.0.bn2.running_mean', 'image_encoder.layer1.0.bn2.running_var', 'image_encoder.layer1.0.bn2.num_batches_tracked', 'image_encoder.layer1.1.conv1.weight', 'image_encoder.layer1.1.bn1.weight', 'image_encoder.layer1.1.bn1.bias', 'image_encoder.layer1.1.bn1.running_mean', 'image_encoder.layer1.1.bn1.running_var', 'image_encoder.layer1.1.bn1.num_batches_tracked', 'image_encoder.layer1.1.

In [ ]:

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=13, pretrained=False):
        super().__init__()

        # Load architecture only; checkpoint supplies trained weights.
        self.image_encoder = resnet18(
            weights=ResNet18_Weights.DEFAULT if pretrained else None
        )

        image_features = self.image_encoder.fc.in_features
        self.image_encoder.fc = nn.Identity()

        self.image_projection = nn.Sequential(
            nn.Linear(image_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.text_projection = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, images, text_embeddings):
        image_features = self.image_projection(
            self.image_encoder(images)
        )
        text_features = self.text_projection(text_embeddings)

        fused = torch.cat([image_features, text_features], dim=1)
        return self.classifier(fused)


global_model = MultimodalClassifier(num_classes=13).to(device)

state_dict = torch.load(
    E2_DIR / "checkpoints/best.pt",
    map_location="cpu",
    weights_only=False
)

load_result = global_model.load_state_dict(state_dict, strict=True)
global_model.eval()

print("Checkpoint loaded successfully.")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# Verify output dimensions.
with torch.no_grad():
    dummy_images = torch.randn(2, 3, 224, 224, device=device)
    dummy_text = torch.randn(2, 768, device=device)
    dummy_logits = global_model(dummy_images, dummy_text)

print("Forward-pass output shape:", tuple(dummy_logits.shape))
assert dummy_logits.shape == (2, 13)


Checkpoint loaded successfully.
Missing keys: []
Unexpected keys: []
Forward-pass output shape: (2, 13)


In [ ]:

import numpy as np
import torch
from torch.utils.data import DataLoader

# Load and align cached text embeddings by sample ID.
embedding_sample_ids = [
    int(x) for x in embedding_data["sample_ids"]
]
text_embeddings = embedding_data["embeddings"].float().cpu()

embedding_by_id = {
    sample_id: text_embeddings[i]
    for i, sample_id in enumerate(embedding_sample_ids)
}

assert len(embedding_by_id) == len(embedding_sample_ids)

# Confirm all experiment samples have corresponding text embeddings.
all_required_ids = set(train_rows["sample_id"].astype(int))
all_required_ids |= set(val_rows["sample_id"].astype(int))

missing_embedding_ids = all_required_ids - set(embedding_by_id)
assert not missing_embedding_ids, (
    f"Missing text embeddings for {len(missing_embedding_ids)} samples"
)

print("Text embedding alignment: PASS")
print("Required sample IDs:", len(all_required_ids))

# Create one dataset per client.
client_loaders = {}
BATCH_SIZE = 16
NUM_WORKERS = 2

for client_id, sample_ids in client_sample_ids.items():
    dataset = PreprocessedCachedImageDataset(
        manifest_path=MANIFEST_PATH,
        cache_root=Path("/content/dp_mmfl_image_cache_256/train"),
        sample_ids=sample_ids,
        image_transform=image_eval_transform,
    )

    client_loaders[client_id] = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

    print(
        f"{client_id}: {len(dataset)} samples, "
        f"{len(client_loaders[client_id])} batches"
    )

print("\nClient DataLoaders created.")


Text embedding alignment: PASS
Required sample IDs: 1000
client_1: 257 samples, 17 batches
client_2: 314 samples, 20 batches
client_3: 264 samples, 17 batches

Client DataLoaders created.


In [ ]:
from pathlib import Path
from torchvision import transforms
from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset

CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")

# Define transforms if they aren't already defined in the current namespace
image_train_transform = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

image_eval_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

print("PreprocessedCachedImageDataset and transforms imported successfully.")

PreprocessedCachedImageDataset and transforms imported successfully.


In [ ]:
!pip install -q redivis transformers pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.2/862.2 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.4/407.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from io import BytesIO
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
import redivis

CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT_256.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(MANIFEST_PATH)

# Combine all IDs needed by train and val
all_needed_ids = set(train_sample_ids) | set(val_sample_ids)
existing_ids = {int(p.stem) for p in CACHE_ROOT_256.glob("*.png")}
missing_ids = all_needed_ids - existing_ids

print(f"Total needed samples: {len(all_needed_ids)}")
print(f"Already cached:       {len(existing_ids & all_needed_ids)}")
print(f"Missing from cache:   {len(missing_ids)}")

if missing_ids:
    print(f"\nFetching {len(missing_ids)} missing images from Redivis...")
    table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")
    missing_manifest = manifest[manifest["sample_id"].isin(missing_ids)].copy()

    for row in tqdm(missing_manifest.itertuples(), total=len(missing_manifest), desc="Downloading missing images"):
        dst_path = CACHE_ROOT_256 / f"{int(row.sample_id):07d}.png"
        if dst_path.exists():
            continue

        png_path = str(row.path_to_image)
        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]
        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"

        try:
            file = table.file(png_path)
            data = file.read(as_text=False)
            img = Image.open(BytesIO(data)).convert("L")
            img = img.resize((256, 256), Image.Resampling.BILINEAR)
            img.save(dst_path, format="PNG")
        except Exception as e:
            print(f"Failed sample {row.sample_id}: {e}")

# Verify that 100% of required experiment images exist
still_missing = all_needed_ids - {int(p.stem) for p in CACHE_ROOT_256.glob("*.png")}
assert len(still_missing) == 0, f"Still missing {len(still_missing)} images: {still_missing}"
print("\nAll required experiment images verified in local cache: PASS")

Total needed samples: 1000
Already cached:       0
Missing from cache:   1000

Fetching 1000 missing images from Redivis...


Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=851f136118d42b83072bd08f0f9c449e

All required experiment images verified in local cache: PASS


In [ ]:
def train_client(model, loader, optimizer=None):
    model.train()
    total_loss = 0.0
    total_valid_labels = 0

    if optimizer is None:
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    for batch in loader:
        # Support both dictionary and tuple data formats
        if isinstance(batch, dict):
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True).float()
            masks = batch["label_mask"].to(device, non_blocking=True).float()
            ids = [int(x) for x in batch["sample_id"]]
        else:
            images, labels, masks, ids, _ = batch
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()
            masks = masks.to(device, non_blocking=True).float()
            ids = [int(x) for x in ids]

        text_batch = torch.stack([embedding_by_id[sid] for sid in ids]).to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images, text_batch)
        loss = masked_bce_with_logits(logits, labels, masks)
        loss.backward()
        optimizer.step()

        batch_valid = masks.sum().item()
        total_loss += loss.item() * batch_valid
        total_valid_labels += batch_valid

    return total_loss / max(total_valid_labels, 1)

In [ ]:
def evaluate_global(model, loader):
    model.eval()
    total_loss = 0.0
    total_valid_labels = 0

    all_logits = []
    all_targets = []
    all_masks = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True).float()
            masks = batch["label_mask"].to(device, non_blocking=True).float()

            ids = batch["sample_id"]
            if torch.is_tensor(ids) or isinstance(ids, np.ndarray):
                ids = ids.tolist()
            ids = [int(x) for x in ids]

            text_batch = torch.stack([embedding_by_id[sid] for sid in ids]).to(device, non_blocking=True)

            logits = model(images, text_batch)
            loss = masked_bce_with_logits(logits, labels, masks)

            batch_valid = masks.sum().item()
            total_loss += loss.item() * batch_valid
            total_valid_labels += batch_valid

            all_logits.append(logits.cpu())
            all_targets.append(labels.cpu())
            all_masks.append(masks.cpu())

    avg_loss = total_loss / max(total_valid_labels, 1)

    all_probs = torch.sigmoid(torch.cat(all_logits, dim=0)).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_masks = torch.cat(all_masks, dim=0).numpy()

    per_class_auroc = {}
    valid_aurocs = []

    for i, col in enumerate(TARGET_COLUMNS):
        col_mask = all_masks[:, i] == 1.0
        y_true = all_targets[col_mask, i]
        y_pred = all_probs[col_mask, i]

        if len(np.unique(y_true)) == 2:
            score = roc_auc_score(y_true, y_pred)
            per_class_auroc[col] = float(score)
            valid_aurocs.append(score)
        else:
            per_class_auroc[col] = None

    macro_auroc = float(np.mean(valid_aurocs)) if valid_aurocs else None
    return avg_loss, macro_auroc, per_class_auroc

In [ ]:
def train_client(model, loader, optimizer=None):
    model.train()
    total_loss = 0.0
    total_valid_labels = 0

    if optimizer is None:
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    for batch in loader:
        # Access batch values via dictionary keys
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True).float()
        masks = batch["label_mask"].to(device, non_blocking=True).float()

        # Ensure sample IDs are integers for the embedding lookup table
        ids = batch["sample_id"]
        if torch.is_tensor(ids) or isinstance(ids, np.ndarray):
            ids = ids.tolist()
        ids = [int(x) for x in ids]

        # Stack aligned text embeddings
        text_batch = torch.stack([embedding_by_id[sid] for sid in ids]).to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images, text_batch)
        loss = masked_bce_with_logits(logits, labels, masks)
        loss.backward()
        optimizer.step()

        batch_valid = masks.sum().item()
        total_loss += loss.item() * batch_valid
        total_valid_labels += batch_valid

    return total_loss / max(total_valid_labels, 1)

In [ ]:
import copy
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from dp_mmfl.data.labels import TARGET_COLUMNS
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# ---------- Parameters & Directories ----------
NUM_ROUNDS = 5
LOCAL_EPOCHS = 1
LEARNING_RATE = 1e-4

CHECKPOINT_DIR = E3_DIR / "checkpoints"
METRICS_DIR = E3_DIR / "metrics"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Local Client Training Helper ----------
def train_client(model, loader):
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4,
    )
    total_loss = 0.0
    total_valid_labels = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True).float()
        masks = batch["label_mask"].to(device, non_blocking=True).float()

        ids = batch["sample_id"]
        if torch.is_tensor(ids) or isinstance(ids, np.ndarray):
            ids = ids.tolist()
        ids = [int(x) for x in ids]

        text_batch = torch.stack(
            [embedding_by_id[sid] for sid in ids]
        ).to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images, text_batch)
        loss = masked_bce_with_logits(logits, labels, masks)
        loss.backward()
        optimizer.step()

        batch_valid = masks.sum().item()
        total_loss += loss.item() * batch_valid
        total_valid_labels += batch_valid

    return total_loss / max(total_valid_labels, 1)

# ---------- Central Evaluation Helper ----------
def evaluate_global(model, loader):
    model.eval()
    total_loss = 0.0
    total_valid_labels = 0

    all_logits = []
    all_targets = []
    all_masks = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True).float()
            masks = batch["label_mask"].to(device, non_blocking=True).float()

            ids = batch["sample_id"]
            if torch.is_tensor(ids) or isinstance(ids, np.ndarray):
                ids = ids.tolist()
            ids = [int(x) for x in ids]

            text_batch = torch.stack(
                [embedding_by_id[sid] for sid in ids]
            ).to(device, non_blocking=True)

            logits = model(images, text_batch)
            loss = masked_bce_with_logits(logits, labels, masks)

            batch_valid = masks.sum().item()
            total_loss += loss.item() * batch_valid
            total_valid_labels += batch_valid

            all_logits.append(logits.cpu())
            all_targets.append(labels.cpu())
            all_masks.append(masks.cpu())

    avg_loss = total_loss / max(total_valid_labels, 1)

    all_probs = torch.sigmoid(torch.cat(all_logits, dim=0)).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_masks = torch.cat(all_masks, dim=0).numpy()

    per_class_auroc = {}
    valid_aurocs = []

    for i, col in enumerate(TARGET_COLUMNS):
        col_mask = all_masks[:, i] == 1.0
        y_true = all_targets[col_mask, i]
        y_pred = all_probs[col_mask, i]

        if len(np.unique(y_true)) == 2:
            score = roc_auc_score(y_true, y_pred)
            per_class_auroc[col] = float(score)
            valid_aurocs.append(score)
        else:
            per_class_auroc[col] = None

    macro_auroc = float(np.mean(valid_aurocs)) if valid_aurocs else None
    return avg_loss, macro_auroc, per_class_auroc

# ---------- FedAvg Aggregator ----------
def fed_avg(client_models, client_counts):
    total_samples = sum(client_counts.values())
    aggregated_state = copy.deepcopy(client_models[0].state_dict())

    # Zero out only floating-point weights and biases
    for k, v in aggregated_state.items():
        if v.is_floating_point():
            aggregated_state[k] = torch.zeros_like(v)

    # Sample-weighted accumulation for floats; copy non-floats
    for model, (client_id, count) in zip(client_models, client_counts.items()):
        weight = count / total_samples
        state = model.state_dict()
        for k, v in state.items():
            if v.is_floating_point():
                aggregated_state[k] += v * weight
            else:
                # Keep tracking counters (e.g. num_batches_tracked) intact
                aggregated_state[k] = v

    return aggregated_state

# ---------- Global Model Setup & Baseline Evaluation ----------
global_model = MultimodalClassifier(num_classes=13, pretrained=True).to(device)

e2_checkpoint_path = PROJECT_ROOT / "results" / "E2_multimodal" / "checkpoints" / "best.pt"
if not e2_checkpoint_path.exists():
    e2_checkpoint_path = PROJECT_ROOT / "results" / "E2_multimodal" / "checkpoints" / "best_model.pt"

print(f"Loading base weights from: {e2_checkpoint_path}")
global_model.load_state_dict(torch.load(e2_checkpoint_path, map_location=device), strict=True)

init_loss, init_auroc, _ = evaluate_global(global_model, val_loader)
auroc_str = f"{init_auroc:.4f}" if init_auroc is not None else "N/A"
print(f"Round 00 (Initial) | Global Val Loss: {init_loss:.4f} | Global Macro AUROC: {auroc_str}\n")

# ---------- Training Execution ----------
client_sample_counts = {cid: len(sids) for cid, sids in client_sample_ids.items()}
best_val_loss = float("inf")
federated_history = []

for round_num in range(1, NUM_ROUNDS + 1):
    round_start = time.perf_counter()
    client_models = []
    client_train_losses = {}

    for client_id, loader in client_loaders.items():
        local_model = copy.deepcopy(global_model).to(device)
        for _ in range(LOCAL_EPOCHS):
            loss = train_client(local_model, loader)
        client_train_losses[client_id] = loss
        client_models.append(local_model)

    # Server aggregation
    global_model.load_state_dict(fed_avg(client_models, client_sample_counts))

    # Server evaluation
    val_loss, val_macro_auroc, per_class_auroc = evaluate_global(global_model, val_loader)
    duration = time.perf_counter() - round_start

    mean_train_loss = float(np.mean(list(client_train_losses.values())))
    auroc_str = f"{val_macro_auroc:.4f}" if val_macro_auroc is not None else "N/A"

    print(
        f"Round {round_num:02d}/{NUM_ROUNDS:02d} [{duration:.1f}s] | "
        f"Avg Client Train Loss: {mean_train_loss:.4f} | "
        f"Global Val Loss: {val_loss:.4f} | "
        f"Global Val Macro AUROC: {auroc_str}"
    )

    record = {
        "round": round_num,
        "mean_client_train_loss": mean_train_loss,
        "client_train_losses": client_train_losses,
        "global_val_loss": val_loss,
        "global_val_macro_auroc": val_macro_auroc,
        "per_class_auroc": per_class_auroc,
        "duration_sec": duration,
    }
    federated_history.append(record)

    # Save per-round checkpoint
    torch.save(
        {
            "round": round_num,
            "global_model_state_dict": global_model.state_dict(),
            "global_val_loss": val_loss,
            "global_val_macro_auroc": val_macro_auroc,
            "client_train_losses": client_train_losses,
        },
        CHECKPOINT_DIR / f"round_{round_num:02d}.pt",
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(global_model.state_dict(), CHECKPOINT_DIR / "best.pt")
        print(f"  --> Saved new best global model (Val Loss: {val_loss:.4f})")

torch.save(global_model.state_dict(), CHECKPOINT_DIR / "last.pt")

# Save metric histories
with open(METRICS_DIR / "training_history.json", "w") as f:
    json.dump(federated_history, f, indent=2)

flat_records = []
for r in federated_history:
    row = {
        "round": r["round"],
        "mean_client_train_loss": r["mean_client_train_loss"],
        "global_val_loss": r["global_val_loss"],
        "global_val_macro_auroc": r["global_val_macro_auroc"],
        "duration_sec": r["duration_sec"],
    }
    for cid, l in r["client_train_losses"].items():
        row[f"loss_{cid}"] = l
    if r["per_class_auroc"]:
        for k, v in r["per_class_auroc"].items():
            row[f"auroc_{k}"] = v
    flat_records.append(row)

pd.DataFrame(flat_records).to_csv(METRICS_DIR / "training_history.csv", index=False)
print("\nE3 FedAvg execution complete. All artifacts saved to Drive.")

Loading base weights from: /content/drive/MyDrive/DP-MMFL/results/E2_multimodal/checkpoints/best.pt
Round 00 (Initial) | Global Val Loss: 0.4720 | Global Macro AUROC: 0.4738

Round 01/05 [4.0s] | Avg Client Train Loss: 0.4005 | Global Val Loss: 0.4775 | Global Val Macro AUROC: 0.5031
  --> Saved new best global model (Val Loss: 0.4775)
Round 02/05 [4.0s] | Avg Client Train Loss: 0.3812 | Global Val Loss: 0.5013 | Global Val Macro AUROC: 0.5432
Round 03/05 [4.0s] | Avg Client Train Loss: 0.3562 | Global Val Loss: 0.5005 | Global Val Macro AUROC: 0.5377
Round 04/05 [4.6s] | Avg Client Train Loss: 0.3200 | Global Val Loss: 0.5297 | Global Val Macro AUROC: 0.5757
Round 05/05 [4.2s] | Avg Client Train Loss: 0.2891 | Global Val Loss: 0.5132 | Global Val Macro AUROC: 0.5424

E3 FedAvg execution complete. All artifacts saved to Drive.


In [ ]:
import json
import torch
import pandas as pd
from pathlib import Path

print("E3 artifact verification")
print("-" * 40)

# Check for round_metrics.csv or training_history.csv
metrics_path = METRICS_DIR / "round_metrics.csv"
if not metrics_path.exists():
    metrics_path = METRICS_DIR / "training_history.csv"

best_path = CHECKPOINT_DIR / "best.pt"

metrics_df = pd.read_csv(metrics_path)
print("Metrics rows:", len(metrics_df))
print(metrics_df.to_string(index=False))

print("\nCheckpoint files:")
for path in sorted(CHECKPOINT_DIR.glob("*.pt")):
    print(f"{path.name:15s}: {path.stat().st_size / (1024**2):.2f} MB")

best_checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)

# Support state_dict or metadata dict formats
if isinstance(best_checkpoint, dict) and "global_model_state_dict" in best_checkpoint:
    print("\nBest checkpoint keys:   ", list(best_checkpoint.keys()))
    print("Best checkpoint round:  ", best_checkpoint.get("round"))
    print("Best checkpoint val loss:", best_checkpoint.get("global_val_loss"))
elif isinstance(best_checkpoint, dict) and "model_state_dict" in best_checkpoint:
    print("\nBest checkpoint keys:   ", list(best_checkpoint.keys()))
    print("Best checkpoint round:  ", best_checkpoint.get("round"))
    print("Best checkpoint metrics:", best_checkpoint.get("metrics"))
else:
    print("\nBest checkpoint format:  Raw model state_dict")
    print("Total parameter tensors: ", len(best_checkpoint))

assert len(metrics_df) == 5, f"Expected 5 rounds in metrics, found {len(metrics_df)}"
assert best_path.exists(), "best.pt is missing!"
assert (CHECKPOINT_DIR / "round_05.pt").exists(), "round_05.pt is missing!"

print("\nE3 artifact verification: PASS")

E3 artifact verification
----------------------------------------
Metrics rows: 5
 round  mean_client_train_loss  global_val_loss  global_val_macro_auroc  duration_sec  loss_client_1  loss_client_2  loss_client_3  auroc_Enlarged Cardiomediastinum  auroc_Cardiomegaly  auroc_Lung Opacity  auroc_Lung Lesion  auroc_Edema  auroc_Consolidation  auroc_Pneumonia  auroc_Atelectasis  auroc_Pneumothorax  auroc_Pleural Effusion  auroc_Pleural Other  auroc_Fracture  auroc_Support Devices
     1                0.400498         0.477491                0.503109      3.952316       0.380761       0.423850       0.396884                          0.625000            0.531429            0.417647             0.5000     0.494118             0.453333         0.390625           0.085714            0.495652                0.505785                  NaN        0.636905               0.901099
     2                0.381152         0.501289                0.543163      4.046802       0.363060       0.397840       

DP

In [ ]:
import importlib.util

if importlib.util.find_spec("opacus") is None:
    print("Opacus is NOT installed.")
else:
    import opacus
    print("Opacus version:", opacus.__version__)

Opacus version: 1.6.0


In [ ]:
!pip install -q opacus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 9.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import torch
from opacus.validators import ModuleValidator

E4_DIR = PROJECT_ROOT / "results" / "E4_dp_federated"
E4_CHECKPOINT_DIR = E4_DIR / "checkpoints"
E4_METRICS_DIR = E4_DIR / "metrics"
E4_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
E4_METRICS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load the original centralized multimodal weights
e2_checkpoint_path = E2_DIR / "checkpoints" / "best.pt"
if not e2_checkpoint_path.exists():
    e2_checkpoint_path = E2_DIR / "checkpoints" / "best_model.pt"

e2_state = torch.load(e2_checkpoint_path, map_location="cpu", weights_only=False)

# 2. Inspect the original architecture for DP-SGD compatibility issues
original_model = MultimodalClassifier(num_classes=13)
original_errors = ModuleValidator.validate(original_model, strict=False)

print("=== Opacus Compatibility Check ===")
print("Original model validation issues found:", len(original_errors))
for error in original_errors:
    print(" -", error)

# 3. Apply ModuleValidator fix (replaces BatchNorm2d with GroupNorm)
dp_model = ModuleValidator.fix(original_model)

# Verify no remaining invalid layers
post_fix_errors = ModuleValidator.validate(dp_model, strict=False)
print("\nPost-fix validation issues remaining:", len(post_fix_errors))
assert len(post_fix_errors) == 0, "Model still contains incompatible layers for Opacus!"

# 4. Transfer compatible pretrained weights
load_result = dp_model.load_state_dict(e2_state, strict=False)
print("\n=== Parameter Transfer Summary ===")
print("Missing keys (e.g., converted GroupNorm layers):", len(load_result.missing_keys))
print("Unexpected keys (e.g., discarded BatchNorm buffers):", len(load_result.unexpected_keys))

print("\nSample missing keys:", load_result.missing_keys[:4])
print("Sample unexpected keys:", load_result.unexpected_keys[:4])
print("\nE4 model preparation: PASS")

=== Opacus Compatibility Check ===
Original model validation issues found: 20
 - BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its peers in a batch, ie the same sample x will get normalized to a different value depending on who else is on its batch. Privacy-wise, this means that we would have to put a privacy mechanism there too. While it can in principle be done, there are now multiple normalization layers that do not have this issue: LayerNorm, InstanceNorm and their generalization GroupNorm are all privacy-safe since they don't have this property.We offer utilities to automatically replace BatchNorms to GroupNorms and we will release pretrained models to help transition, such as GN-ResNet ie a ResNet using GroupNorm, pretrained on ImageNet
 - BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on it

In [ ]:

import inspect
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

print("PrivacyEngine.make_private signature:")
print(inspect.signature(PrivacyEngine.make_private))

print("\nPrivacyEngine.get_epsilon signature:")
print(inspect.signature(PrivacyEngine.get_epsilon))

print("\nDP model validation:")
print("Issues:", len(ModuleValidator.validate(dp_model, strict=False)))


PrivacyEngine.make_private signature:
(self, *, module: torch.nn.modules.module.Module, optimizer: torch.optim.optimizer.Optimizer, criterion=CrossEntropyLoss(), data_loader: torch.utils.data.dataloader.DataLoader, noise_multiplier: float, max_grad_norm: Union[float, List[float]], batch_first: bool = True, loss_reduction: str = 'mean', poisson_sampling: bool = True, clipping: str = 'flat', noise_generator=None, grad_sample_mode: str = 'hooks', wrap_model: bool = True, rand_on_empty: bool = False, **kwargs) -> Union[Tuple[Union[opacus.grad_sample.gsm_base.AbstractGradSampleModule, opacus.grad_sample.grad_sample_module.GradSampleHooks], opacus.optimizers.optimizer.DPOptimizer, torch.utils.data.dataloader.DataLoader], Tuple[Union[opacus.grad_sample.gsm_base.AbstractGradSampleModule, opacus.grad_sample.grad_sample_module.GradSampleHooks], opacus.optimizers.optimizer.DPOptimizer, opacus.utils.fast_gradient_clipping_utils.DPLossFastGradientClipping, torch.utils.data.dataloader.DataLoader]]



In [ ]:

import copy
import json
import torch
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

# E4 pilot configuration
SMOKE_CLIENT = "client_1"
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
DELTA = 1e-5
LEARNING_RATE_DP = 1e-4

# Create a fresh model from the DP-compatible E2 initialization.
smoke_model = MultimodalClassifier(num_classes=13)
smoke_model = ModuleValidator.fix(smoke_model)

# Transfer the E2 learned weights into the compatible architecture.
smoke_model.load_state_dict(e2_state, strict=False)
smoke_model = smoke_model.to(device)

optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=LEARNING_RATE_DP,
)

privacy_engine = PrivacyEngine(accountant="rdp")

private_model, private_optimizer, private_loader = (
    privacy_engine.make_private(
        module=smoke_model,
        optimizer=optimizer,
        data_loader=client_loaders[SMOKE_CLIENT],
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=MAX_GRAD_NORM,
        poisson_sampling=True,
        loss_reduction="mean",
    )
)

print("Private model prepared.")
print("Private batches:", len(private_loader))

# Run a single private optimizer step.
private_model.train()
batch = next(iter(private_loader))
images, labels, masks, sample_ids, _patient_ids = batch

images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True).float()
masks = masks.to(device, non_blocking=True).float()
texts = get_batch_text(sample_ids).to(device, non_blocking=True)

private_optimizer.zero_grad(set_to_none=True)
logits = private_model(images, texts)
loss = federated_masked_bce(logits, labels, masks)
loss.backward()
private_optimizer.step()

epsilon_after_step = privacy_engine.get_epsilon(delta=DELTA)

print(f"Loss after one private step: {loss.item():.4f}")
print(f"Accounted epsilon after one step: {epsilon_after_step:.4f}")
print(f"Delta: {DELTA}")
print("DP optimizer step: PASS")

# Save smoke-test artifacts.
torch.save(
    {
        "model_state_dict": private_model._module.state_dict()
            if hasattr(private_model, "_module")
            else private_model.state_dict(),
        "noise_multiplier": NOISE_MULTIPLIER,
        "max_grad_norm": MAX_GRAD_NORM,
        "delta": DELTA,
        "epsilon_after_one_step": epsilon_after_step,
        "client": SMOKE_CLIENT,
        "note": "One-step DP-SGD smoke test; not a final model.",
    },
    E4_CHECKPOINT_DIR / "dp_smoke_test.pt",
)

with open(E4_METRICS_DIR / "dp_smoke_test.json", "w") as f:
    json.dump(
        {
            "client": SMOKE_CLIENT,
            "loss_after_one_step": float(loss.item()),
            "epsilon_after_one_step": float(epsilon_after_step),
            "delta": DELTA,
            "noise_multiplier": NOISE_MULTIPLIER,
            "max_grad_norm": MAX_GRAD_NORM,
            "status": "passed",
        },
        f,
        indent=2,
    )

print("Smoke-test artifacts saved.")


Private model prepared.
Private batches: 17


AttributeError: 'str' object has no attribute 'to'

In [ ]:
# E4 — Step 3: Run Private-Training Smoke Test
import copy
import json
import time
import torch
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# E4 pilot configuration
SMOKE_CLIENT = "client_1"
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
DELTA = 1e-5
LEARNING_RATE_DP = 1e-4

print(f"=== Running DP-SGD Smoke Test on {SMOKE_CLIENT} ===")

# 1. Fresh DP-compatible model initialized with E2 weights
smoke_model = MultimodalClassifier(num_classes=13)
smoke_model = ModuleValidator.fix(smoke_model)
smoke_model.load_state_dict(e2_state, strict=False)
smoke_model = smoke_model.to(device)

optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=LEARNING_RATE_DP,
)

# 2. Attach Opacus PrivacyEngine (RDP accountant)
privacy_engine = PrivacyEngine(accountant="rdp")

private_model, private_optimizer, private_loader = privacy_engine.make_private(
    module=smoke_model,
    optimizer=optimizer,
    data_loader=client_loaders[SMOKE_CLIENT],
    noise_multiplier=NOISE_MULTIPLIER,
    max_grad_norm=MAX_GRAD_NORM,
)

# 3. Execute 1 Local Epoch of DP-SGD
private_model.train()
smoke_loss_accum = 0.0
valid_label_count = 0
start_time = time.perf_counter()

for batch_idx, batch in enumerate(private_loader):
    images = batch["image"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True).float()
    masks = batch["label_mask"].to(device, non_blocking=True).float()

    ids = batch["sample_id"]
    if torch.is_tensor(ids) or hasattr(ids, "numpy"):
        ids = ids.tolist()
    ids = [int(x) for x in ids]

    text_batch = torch.stack(
        [embedding_by_id[sid] for sid in ids]
    ).to(device, non_blocking=True)

    private_optimizer.zero_grad()
    logits = private_model(images, text_batch)
    loss = masked_bce_with_logits(logits, labels, masks)
    loss.backward()
    private_optimizer.step()

    valid_count = masks.sum().item()
    smoke_loss_accum += loss.item() * valid_count
    valid_label_count += valid_count

elapsed = time.perf_counter() - start_time
avg_smoke_loss = smoke_loss_accum / max(valid_label_count, 1)

# 4. Compute Cumulative Epsilon
epsilon = privacy_engine.get_epsilon(delta=DELTA)

print(f"Smoke Test Duration: {elapsed:.2f}s")
print(f"Average Batch Loss:  {avg_smoke_loss:.4f}")
print(f"Privacy Budget:      ε = {epsilon:.4f}, δ = {DELTA}")
print(f"DP-SGD Verification: PASS")

# 5. Save Smoke Test Artifact
smoke_artifact = {
    "smoke_client": SMOKE_CLIENT,
    "noise_multiplier": NOISE_MULTIPLIER,
    "max_grad_norm": MAX_GRAD_NORM,
    "delta": DELTA,
    "epsilon": float(epsilon),
    "loss": float(avg_smoke_loss),
}

with open(E4_METRICS_DIR / "smoke_test_summary.json", "w") as f:
    json.dump(smoke_artifact, f, indent=2)

print(f"Artifact saved:      {E4_METRICS_DIR / 'smoke_test_summary.json'}")

=== Running DP-SGD Smoke Test on client_1 ===


RuntimeError: Output 0 of BackwardHookFunctionBackward is a view and is being modified inplace. This view was created inside a custom Function (or because an input was returned as-is) and the autograd logic to handle view+inplace would override the custom backward associated with the custom Function, leading to incorrect gradients. This behavior is forbidden. You can fix this by cloning the output of the custom Function.

In [ ]:
import copy
import json
import time
import torch
import torch.nn as nn
from torchvision.models.resnet import BasicBlock
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# 1. Patch BasicBlock.forward to perform out-of-place residual addition
def safe_basicblock_forward(self, x):
    identity = x

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    if self.downsample is not None:
        identity = self.downsample(x)

    # Out-of-place addition solves the autograd view+inplace conflict
    out = out + identity
    out = self.relu(out)

    return out

BasicBlock.forward = safe_basicblock_forward

# 2. Configuration
SMOKE_CLIENT = "client_1"
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
DELTA = 1e-5
LEARNING_RATE_DP = 1e-4

print(f"=== Running DP-SGD Smoke Test on {SMOKE_CLIENT} (Patched ResNet) ===")

# 3. Model setup
smoke_model = MultimodalClassifier(num_classes=13)
smoke_model = ModuleValidator.fix(smoke_model)

# Disable in-place ReLUs recursively
for module in smoke_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

smoke_model.load_state_dict(e2_state, strict=False)
smoke_model = smoke_model.to(device)

optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=LEARNING_RATE_DP,
)

# 4. Attach Opacus PrivacyEngine
privacy_engine = PrivacyEngine(accountant="rdp")

private_model, private_optimizer, private_loader = privacy_engine.make_private(
    module=smoke_model,
    optimizer=optimizer,
    data_loader=client_loaders[SMOKE_CLIENT],
    noise_multiplier=NOISE_MULTIPLIER,
    max_grad_norm=MAX_GRAD_NORM,
)

# 5. Run 1 Local Epoch
private_model.train()
smoke_loss_accum = 0.0
valid_label_count = 0
start_time = time.perf_counter()

for batch in private_loader:
    images = batch["image"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True).float()
    masks = batch["label_mask"].to(device, non_blocking=True).float()

    ids = batch["sample_id"]
    if torch.is_tensor(ids) or hasattr(ids, "numpy"):
        ids = ids.tolist()
    ids = [int(x) for x in ids]

    text_batch = torch.stack(
        [embedding_by_id[sid] for sid in ids]
    ).to(device, non_blocking=True)

    private_optimizer.zero_grad()
    logits = private_model(images, text_batch)
    loss = masked_bce_with_logits(logits, labels, masks)
    loss.backward()
    private_optimizer.step()

    valid_count = masks.sum().item()
    smoke_loss_accum += loss.item() * valid_count
    valid_label_count += valid_count

elapsed = time.perf_counter() - start_time
avg_smoke_loss = smoke_loss_accum / max(valid_label_count, 1)

# 6. Privacy Accounting
epsilon = privacy_engine.get_epsilon(delta=DELTA)

print(f"Smoke Test Duration: {elapsed:.2f}s")
print(f"Average Batch Loss:  {avg_smoke_loss:.4f}")
print(f"Privacy Budget:      ε = {epsilon:.4f}, δ = {DELTA}")
print("DP-SGD Verification: PASS")

# 7. Save Artifact
smoke_artifact = {
    "smoke_client": SMOKE_CLIENT,
    "noise_multiplier": NOISE_MULTIPLIER,
    "max_grad_norm": MAX_GRAD_NORM,
    "delta": DELTA,
    "epsilon": float(epsilon),
    "loss": float(avg_smoke_loss),
    "duration_sec": elapsed,
}

with open(E4_METRICS_DIR / "smoke_test_summary.json", "w") as f:
    json.dump(smoke_artifact, f, indent=2)

print(f"Artifact saved:      {E4_METRICS_DIR / 'smoke_test_summary.json'}")

=== Running DP-SGD Smoke Test on client_1 (Patched ResNet) ===
Smoke Test Duration: 4.83s
Average Batch Loss:  0.4005
Privacy Budget:      ε = 2.6842, δ = 1e-05
DP-SGD Verification: PASS
Artifact saved:      /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/metrics/smoke_test_summary.json


In [ ]:
# E4 Step 4: 1-Round Pilot Federated DP-SGD Training Across 3 Clients
import copy
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock
from dp_mmfl.data.labels import TARGET_COLUMNS
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# Ensure out-of-place residual addition patch is active for Opacus autograd compatibility
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

# ---------- E4 Pilot Configuration ----------
E4_ROUNDS = 1
E4_LOCAL_EPOCHS = 1
E4_NOISE_MULTIPLIER = 1.0
E4_MAX_GRAD_NORM = 1.0
E4_DELTA = 1e-5
E4_LR = 1e-4

E4_DIR = PROJECT_ROOT / "results" / "E4_dp_federated"
E4_CHECKPOINT_DIR = E4_DIR / "checkpoints"
E4_METRICS_DIR = E4_DIR / "metrics"
E4_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
E4_METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Helper to instantiate and adapt DP-compatible architecture
def make_dp_model(global_state):
    model = MultimodalClassifier(num_classes=13)
    model = ModuleValidator.fix(model)
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
    model.load_state_dict(global_state, strict=False)
    return model.to(device)

# Initialize global DP model
global_dp_model = make_dp_model(e2_state)

# Initial baseline evaluation before federated updates
init_val_loss, init_val_auroc, _ = evaluate_global(global_dp_model, val_loader)
auroc_str = f"{init_val_auroc:.4f}" if init_val_auroc is not None else "N/A"
print(f"=== E4 Pilot: Initial Evaluation (Pre-training) ===")
print(f"Global Val Loss: {init_val_loss:.4f} | Global Macro AUROC: {auroc_str}\n")

# ---------- Private Client Training Function ----------
def private_client_train(client_id, global_state):
    model = make_dp_model(global_state)
    optimizer = torch.optim.AdamW(model.parameters(), lr=E4_LR, weight_decay=1e-4)

    privacy_engine = PrivacyEngine(accountant="rdp")
    private_model, private_optimizer, private_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=client_loaders[client_id],
        noise_multiplier=E4_NOISE_MULTIPLIER,
        max_grad_norm=E4_MAX_GRAD_NORM,
    )

    private_model.train()
    loss_accum = 0.0
    valid_labels = 0

    for _ in range(E4_LOCAL_EPOCHS):
        for batch in private_loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True).float()
            masks = batch["label_mask"].to(device, non_blocking=True).float()

            ids = batch["sample_id"]
            if torch.is_tensor(ids) or hasattr(ids, "numpy"):
                ids = ids.tolist()
            ids = [int(x) for x in ids]

            text_batch = torch.stack(
                [embedding_by_id[sid] for sid in ids]
            ).to(device, non_blocking=True)

            private_optimizer.zero_grad()
            logits = private_model(images, text_batch)
            loss = masked_bce_with_logits(logits, labels, masks)
            loss.backward()
            private_optimizer.step()

            valid_count = masks.sum().item()
            loss_accum += loss.item() * valid_count
            valid_labels += valid_count

    avg_loss = loss_accum / max(valid_labels, 1)
    client_eps = privacy_engine.get_epsilon(delta=E4_DELTA)

    # Extract the underlying module's state_dict stripped of Opacus wrapping
    unwrapped_state = private_model._module.state_dict()

    return unwrapped_state, avg_loss, float(client_eps)

# ---------- Safe FedAvg Aggregation ----------
def fed_avg_states(client_states, client_counts):
    total_samples = sum(client_counts.values())
    aggregated_state = copy.deepcopy(client_states[0])

    for k, v in aggregated_state.items():
        if v.is_floating_point():
            aggregated_state[k] = torch.zeros_like(v)

    for state, (client_id, count) in zip(client_states, client_counts.items()):
        weight = count / total_samples
        for k, v in state.items():
            if v.is_floating_point():
                aggregated_state[k] += v * weight
            else:
                aggregated_state[k] = v

    return aggregated_state

# ---------- Execute 1-Round Pilot Across All 3 Clients ----------
client_sample_counts = {cid: len(sids) for cid, sids in client_sample_ids.items()}
round_start = time.perf_counter()

client_states = []
client_losses = {}
client_epsilons = {}

print(f"Starting E4 1-Round Pilot across {len(client_loaders)} clients...")
for client_id in client_loaders.keys():
    c_start = time.perf_counter()
    state, loss, eps = private_client_train(client_id, global_dp_model.state_dict())
    c_time = time.perf_counter() - c_start

    client_states.append(state)
    client_losses[client_id] = loss
    client_epsilons[client_id] = eps
    print(f"  [{client_id}] Done ({c_time:.1f}s) | Train Loss: {loss:.4f} | ε: {eps:.4f} (δ={E4_DELTA})")

# Server FedAvg aggregation
aggregated_weights = fed_avg_states(client_states, client_sample_counts)
global_dp_model.load_state_dict(aggregated_weights)

# Evaluate aggregated global model on central validation set
val_loss, val_macro_auroc, per_class_auroc = evaluate_global(global_dp_model, val_loader)
round_duration = time.perf_counter() - round_start
val_auroc_str = f"{val_macro_auroc:.4f}" if val_macro_auroc is not None else "N/A"

print("\n=== E4 Round 01 Summary ===")
print(f"Round Duration:        {round_duration:.1f}s")
print(f"Mean Client Loss:      {np.mean(list(client_losses.values())):.4f}")
print(f"Global Val Loss:       {val_loss:.4f}")
print(f"Global Macro AUROC:    {val_auroc_str}")
print(f"Per-client Epsilons:   {client_epsilons}")

# Save pilot artifacts
pilot_summary = {
    "round": 1,
    "client_losses": client_losses,
    "client_epsilons": client_epsilons,
    "delta": E4_DELTA,
    "noise_multiplier": E4_NOISE_MULTIPLIER,
    "max_grad_norm": E4_MAX_GRAD_NORM,
    "global_val_loss": val_loss,
    "global_val_macro_auroc": val_macro_auroc,
    "per_class_auroc": per_class_auroc,
    "duration_sec": round_duration,
}

with open(E4_METRICS_DIR / "pilot_round_summary.json", "w") as f:
    json.dump(pilot_summary, f, indent=2)

torch.save(global_dp_model.state_dict(), E4_CHECKPOINT_DIR / "round_01_pilot.pt")
print(f"\nPilot checkpoint & metrics saved under {E4_DIR}")

=== E4 Pilot: Initial Evaluation (Pre-training) ===
Global Val Loss: 0.4634 | Global Macro AUROC: 0.4244

Starting E4 1-Round Pilot across 3 clients...
  [client_1] Done (3.8s) | Train Loss: 0.3911 | ε: 2.6842 (δ=1e-05)
  [client_2] Done (3.2s) | Train Loss: 0.4521 | ε: 2.4813 (δ=1e-05)
  [client_3] Done (3.0s) | Train Loss: 0.4215 | ε: 2.6842 (δ=1e-05)

=== E4 Round 01 Summary ===
Round Duration:        10.4s
Mean Client Loss:      0.4216
Global Val Loss:       0.4646
Global Macro AUROC:    0.4377
Per-client Epsilons:   {'client_1': 2.684195014643082, 'client_2': 2.4812959786270485, 'client_3': 2.684195014643082}

Pilot checkpoint & metrics saved under /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated


In [ ]:
import importlib.util
from pathlib import Path
import opacus

print("Opacus version:        ", opacus.__version__)
print("torchcsprng available: ", importlib.util.find_spec("torchcsprng") is not None)

print("\nE4 output files:")
for path in sorted(E4_DIR.rglob("*")):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f" - {path.relative_to(E4_DIR)} ({size_kb:.1f} KB)")

Opacus version:         1.6.0
torchcsprng available:  False

E4 output files:
 - checkpoints/round_01_pilot.pt (45492.7 KB)
 - metrics/pilot_round_summary.json (1.0 KB)
 - metrics/smoke_test_summary.json (0.2 KB)


In [ ]:
import json

summary_path = E4_DIR / "metrics" / "pilot_round_summary.json"

with open(summary_path, "r") as f:
    pilot_summary = json.load(f)

print(json.dumps(pilot_summary, indent=2))

{
  "round": 1,
  "client_losses": {
    "client_1": 0.3911416024949751,
    "client_2": 0.4521074545217018,
    "client_3": 0.4215451913100613
  },
  "client_epsilons": {
    "client_1": 2.684195014643082,
    "client_2": 2.4812959786270485,
    "client_3": 2.684195014643082
  },
  "delta": 1e-05,
  "noise_multiplier": 1.0,
  "max_grad_norm": 1.0,
  "global_val_loss": 0.46462549159190797,
  "global_val_macro_auroc": 0.4377463488819859,
  "per_class_auroc": {
    "Enlarged Cardiomediastinum": 0.5576923076923077,
    "Cardiomegaly": 0.6342857142857142,
    "Lung Opacity": 0.27941176470588236,
    "Lung Lesion": 0.375,
    "Edema": 0.38088235294117645,
    "Consolidation": 0.4488888888888889,
    "Pneumonia": 0.4375,
    "Atelectasis": 0.5428571428571428,
    "Pneumothorax": 0.5173913043478261,
    "Pleural Effusion": 0.5396694214876032,
    "Pleural Other": null,
    "Fracture": 0.44047619047619047,
    "Support Devices": 0.0989010989010989
  },
  "duration_sec": 10.442536062000272
}


In [ ]:
import inspect
from opacus import PrivacyEngine
from opacus.accountants import RDPAccountant

print("RDPAccountant methods:")
print([name for name in dir(RDPAccountant) if not name.startswith("_")])

print("\nRDPAccountant.step signature:")
print(inspect.signature(RDPAccountant.step))

print("\nPrivacyEngine accountant property:")
print(type(PrivacyEngine().accountant).__name__)

RDPAccountant methods:
['DEFAULT_ALPHAS', 'get_epsilon', 'get_optimizer_hook_fn', 'get_privacy_spent', 'load_state_dict', 'mechanism', 'state_dict', 'step']

RDPAccountant.step signature:
(self, *, noise_multiplier: float, sample_rate: float)

PrivacyEngine accountant property:
PRVAccountant


In [ ]:
import json
from opacus.accountants import RDPAccountant

DELTA = 1e-5
NOISE_MULTIPLIER = 1.0

accounting_test = {}
for client_id, loader in client_loaders.items():
    sample_rate = 1.0 / len(loader)
    accountant = RDPAccountant()
    # Simulate accounting across two rounds with persistent history
    for _ in range(2):
        for _ in range(len(loader)):
            accountant.step(
                noise_multiplier=NOISE_MULTIPLIER,
                sample_rate=sample_rate,
            )
    epsilon = accountant.get_epsilon(delta=DELTA)
    accounting_test[client_id] = {
        "sample_rate_used_for_test": sample_rate,
        "simulated_rounds": 2,
        "simulated_steps_per_round": len(loader),
        "epsilon_after_two_simulated_rounds": float(epsilon),
        "delta": DELTA,
    }

print(json.dumps(accounting_test, indent=2))

{
  "client_1": {
    "sample_rate_used_for_test": 0.058823529411764705,
    "simulated_rounds": 2,
    "simulated_steps_per_round": 17,
    "epsilon_after_two_simulated_rounds": 3.2324517070483907,
    "delta": 1e-05
  },
  "client_2": {
    "sample_rate_used_for_test": 0.05,
    "simulated_rounds": 2,
    "simulated_steps_per_round": 20,
    "epsilon_after_two_simulated_rounds": 2.9702037175605582,
    "delta": 1e-05
  },
  "client_3": {
    "sample_rate_used_for_test": 0.058823529411764705,
    "simulated_rounds": 2,
    "simulated_steps_per_round": 17,
    "epsilon_after_two_simulated_rounds": 3.2324517070483907,
    "delta": 1e-05
  }
}


In [ ]:
# E4 — Step 7: Initialize and Persist Per-Client RDP Accountants (Round 00)
import json
from pathlib import Path
from opacus.accountants import RDPAccountant

E4_ACCOUNTANT_DIR = E4_DIR / "accountants"
E4_ACCOUNTANT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Fresh accounting instances per client
client_accountants = {
    client_id: RDPAccountant()
    for client_id in client_loaders
}

accountant_metadata = {
    "experiment": "E4_federated_DP",
    "accountant": "RDPAccountant",
    "noise_multiplier": E4_NOISE_MULTIPLIER,
    "delta": E4_DELTA,
    "rounds_completed": 0,
    "client_accountants": {},
    "note": (
        "Fresh accounting state. This metadata alone does not establish "
        "a privacy guarantee; actual sampling rates and optimizer steps "
        "must be recorded from the private data loaders."
    ),
}

for client_id, accountant in client_accountants.items():
    eps = accountant.get_epsilon(delta=E4_DELTA)
    accountant_metadata["client_accountants"][client_id] = {
        "state": accountant.state_dict(),
        "epsilon": float(eps) if eps is not None else 0.0,
    }

metadata_path = E4_ACCOUNTANT_DIR / "accountant_state_round_00.json"
with open(metadata_path, "w") as f:
    json.dump(accountant_metadata, f, indent=2)

print("Accountant state directory:", E4_ACCOUNTANT_DIR)
print("Saved initial accounting state:", metadata_path.name)
for cid, acc in client_accountants.items():
    print(f" - {cid}: initialized (ε = {accountant_metadata['client_accountants'][cid]['epsilon']})")

Accountant state directory: /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/accountants
Saved initial accounting state: accountant_state_round_00.json
 - client_1: initialized (ε = 0.0)
 - client_2: initialized (ε = 0.0)
 - client_3: initialized (ε = 0.0)


In [ ]:
import inspect
from opacus import PrivacyEngine

print("PrivacyEngine.make_private:")
print(inspect.signature(PrivacyEngine.make_private))

print("\nPrivacyEngine.accountant:")
engine = PrivacyEngine(accountant="rdp")
print(type(engine.accountant).__name__)
print("State keys:", list(engine.accountant.state_dict().keys()))

print("\nE4 client loaders:")
for client_id, loader in client_loaders.items():
    print(
        f"{client_id}: "
        f"dataset_size = {len(loader.dataset)}, "
        f"batches = {len(loader)}, "
        f"batch_sampler = {type(loader.batch_sampler).__name__}"
    )

PrivacyEngine.make_private:
(self, *, module: torch.nn.modules.module.Module, optimizer: torch.optim.optimizer.Optimizer, criterion=CrossEntropyLoss(), data_loader: torch.utils.data.dataloader.DataLoader, noise_multiplier: float, max_grad_norm: Union[float, List[float]], batch_first: bool = True, loss_reduction: str = 'mean', poisson_sampling: bool = True, clipping: str = 'flat', noise_generator=None, grad_sample_mode: str = 'hooks', wrap_model: bool = True, rand_on_empty: bool = False, **kwargs) -> Union[Tuple[Union[opacus.grad_sample.gsm_base.AbstractGradSampleModule, opacus.grad_sample.grad_sample_module.GradSampleHooks], opacus.optimizers.optimizer.DPOptimizer, torch.utils.data.dataloader.DataLoader], Tuple[Union[opacus.grad_sample.gsm_base.AbstractGradSampleModule, opacus.grad_sample.grad_sample_module.GradSampleHooks], opacus.optimizers.optimizer.DPOptimizer, opacus.utils.fast_gradient_clipping_utils.DPLossFastGradientClipping, torch.utils.data.dataloader.DataLoader]]

PrivacyEng

In [ ]:
# E4 — Step 8: Single-Client Actual Accounting & Step Verification
import json
import time
from pathlib import Path
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# Ensure out-of-place residual patch is in place
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

VERIFY_CLIENT = "client_1"
VERIFY_DIR = E4_DIR / "accounting_verification"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)

# 1. Instantiate Opacus-compatible model from E2 weights
model = MultimodalClassifier(num_classes=13)
model = ModuleValidator.fix(model)
for module in model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

model.load_state_dict(e2_state, strict=False)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
privacy_engine = PrivacyEngine(accountant="rdp")

# 2. Make private with poisson_sampling=True
private_model, private_optimizer, private_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=client_loaders[VERIFY_CLIENT],
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    poisson_sampling=True,
    loss_reduction="mean",
)

# 3. Execute 1 Local Epoch
start = time.perf_counter()
loss_total = 0.0
valid_labels = 0
steps = 0

private_model.train()
for batch in private_loader:
    images = batch["image"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True).float()
    masks = batch["label_mask"].to(device, non_blocking=True).float()

    ids = batch["sample_id"]
    if torch.is_tensor(ids) or hasattr(ids, "numpy"):
        ids = ids.tolist()
    ids = [int(x) for x in ids]

    text_batch = torch.stack(
        [embedding_by_id[sid] for sid in ids]
    ).to(device, non_blocking=True)

    private_optimizer.zero_grad()
    logits = private_model(images, text_batch)
    loss = masked_bce_with_logits(logits, labels, masks)
    loss.backward()
    private_optimizer.step()

    steps += 1
    valid_count = masks.sum().item()
    loss_total += loss.item() * valid_count
    valid_labels += valid_count

elapsed = time.perf_counter() - start
avg_loss = loss_total / max(valid_labels, 1)

# 4. Extract Accountant State & Values
epsilon = privacy_engine.get_epsilon(delta=E4_DELTA)
accountant_state = privacy_engine.accountant.state_dict()

print(f"=== {VERIFY_CLIENT} Accounting Verification ===")
print(f"Duration:               {elapsed:.2f}s")
print(f"Total Steps Taken:      {steps}")
print(f"Average Batch Loss:     {avg_loss:.4f}")
print(f"Computed Privacy (RDP): ε = {epsilon:.4f} (δ = {E4_DELTA})")
print(f"Accountant State Keys:  {list(accountant_state.keys())}")

# 5. Persist Verification Artifacts
verification_summary = {
    "client": VERIFY_CLIENT,
    "steps": steps,
    "elapsed_sec": elapsed,
    "avg_loss": float(avg_loss),
    "epsilon": float(epsilon),
    "delta": E4_DELTA,
    "noise_multiplier": 1.0,
    "max_grad_norm": 1.0,
    "poisson_sampling": True,
    "accountant_type": type(privacy_engine.accountant).__name__,
    "accountant_state": accountant_state,
}

verification_path = VERIFY_DIR / "client_1_accounting_check.json"
with open(verification_path, "w") as f:
    json.dump(verification_summary, f, indent=2)

print(f"\nArtifact successfully saved: {verification_path}")

=== client_1 Accounting Verification ===
Duration:               2.98s
Total Steps Taken:      17
Average Batch Loss:     0.4213
Computed Privacy (RDP): ε = 2.6842 (δ = 1e-05)
Accountant State Keys:  ['history', 'mechanism']

Artifact successfully saved: /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/accounting_verification/client_1_accounting_check.json


In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.accountants import RDPAccountant
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock

# 1. Maintain out-of-place residual addition to avoid autograd view+inplace conflicts
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

ROUND_DIR = E4_DIR / "round_verification"
ROUND_DIR.mkdir(parents=True, exist_ok=True)

DELTA = 1e-5
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
LEARNING_RATE = 1e-4

# 2. Fresh global model initialized from E2
global_model = MultimodalClassifier(num_classes=13)
global_model = ModuleValidator.fix(global_model)
for module in global_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

global_model.load_state_dict(e2_state, strict=False)
global_model = global_model.to(device)

global_state = {
    k: v.detach().cpu().clone()
    for k, v in global_model.state_dict().items()
}

client_states = []
client_sizes = []
client_summaries = []
accountant_states = {}

round_start = time.time()

for client_id, base_loader in client_loaders.items():
    print(f"Training {client_id}...")

    local_model = MultimodalClassifier(num_classes=13)
    local_model = ModuleValidator.fix(local_model)
    for module in local_model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False

    local_model.load_state_dict(global_state, strict=True)
    local_model = local_model.to(device)

    optimizer = torch.optim.AdamW(
        local_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )

    privacy_engine = PrivacyEngine(accountant="rdp")

    private_model, private_optimizer, private_loader = (
        privacy_engine.make_private(
            module=local_model,
            optimizer=optimizer,
            data_loader=base_loader,
            noise_multiplier=NOISE_MULTIPLIER,
            max_grad_norm=MAX_GRAD_NORM,
            poisson_sampling=True,
            loss_reduction="mean",
        )
    )

    private_model.train()
    loss_sum = 0.0
    steps = 0

    for batch in private_loader:
        # Dict unpack safe access
        if isinstance(batch, dict):
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True).float()
            masks = batch["label_mask"].to(device, non_blocking=True).float()
            sids = batch["sample_id"]
            if torch.is_tensor(sids) or isinstance(sids, np.ndarray):
                sids = sids.tolist()
            sids = [int(x) for x in sids]
        else:
            images, labels, masks, sids, _ = batch
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()
            masks = masks.to(device, non_blocking=True).float()
            sids = [int(x) for x in sids]

        texts = torch.stack([embedding_by_id[sid] for sid in sids]).to(device, non_blocking=True)

        private_optimizer.zero_grad(set_to_none=True)
        logits = private_model(images, texts)
        loss = masked_bce_with_logits(logits, labels, masks)
        loss.backward()
        private_optimizer.step()

        loss_sum += float(loss.item())
        steps += 1

    epsilon = privacy_engine.get_epsilon(delta=DELTA)
    accountant_states[client_id] = privacy_engine.accountant.state_dict()

    underlying = (
        private_model._module
        if hasattr(private_model, "_module")
        else private_model
    )
    client_states.append({
        k: v.detach().cpu().clone()
        for k, v in underlying.state_dict().items()
    })
    client_sizes.append(len(base_loader.dataset))

    mean_loss = loss_sum / max(steps, 1)
    client_summaries.append({
        "client": client_id,
        "dataset_size": len(base_loader.dataset),
        "optimizer_steps": steps,
        "mean_batch_loss": mean_loss,
        "epsilon": float(epsilon),
        "delta": DELTA,
        "noise_multiplier": NOISE_MULTIPLIER,
        "max_grad_norm": MAX_GRAD_NORM,
        "accountant": "RDPAccountant",
    })

    print(
        f"   steps={steps} | "
        f"loss={mean_loss:.4f} | "
        f"epsilon={epsilon:.4f}"
    )

    del private_model, private_optimizer, privacy_engine, local_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Sample-weighted FedAvg
total_size = sum(client_sizes)
aggregated_state = {}

for key in client_states[0]:
    tensors = [state[key] for state in client_states]

    if torch.is_floating_point(tensors[0]):
        aggregated_state[key] = sum(
            tensor.float() * (size / total_size)
            for tensor, size in zip(tensors, client_sizes)
        ).to(tensors[0].dtype)
    else:
        # Non-float tracking buffers (e.g., num_batches_tracked)
        aggregated_state[key] = tensors[0].clone()

global_model.load_state_dict(aggregated_state, strict=True)
global_model.eval()

validation = evaluate_global(global_model, val_loader)

summary = {
    "round": 1,
    "validation": {
        "val_loss": float(validation[0]),
        "macro_auroc": float(validation[1]) if validation[1] is not None else None,
        "per_class_auroc": validation[2] if len(validation) > 2 else None,
    },
    "clients": client_summaries,
    "aggregation": "sample_weighted_FedAvg",
    "delta": DELTA,
    "noise_multiplier": NOISE_MULTIPLIER,
    "max_grad_norm": MAX_GRAD_NORM,
    "accountant": "RDPAccountant",
    "secure_rng": False,
    "elapsed_seconds": time.time() - round_start,
    "note": (
        "Experimental image-level DP pilot. This run starts fresh; "
        "client accountants must be restored and continued in later rounds."
    ),
}

torch.save(
    {
        "round": 1,
        "model_state_dict": global_model.state_dict(),
        "summary": summary,
        "accountant_states": accountant_states,
    },
    ROUND_DIR / "round_01.pt",
)

with open(ROUND_DIR / "round_01_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

with open(ROUND_DIR / "accountant_states_round_01.json", "w") as f:
    json.dump(accountant_states, f, indent=2)

print("\nRound 1 complete.")
print("Validation:", summary["validation"])
print("Saved artifacts:", ROUND_DIR)

Training client_1...
   steps=17 | loss=0.3887 | epsilon=2.6842
Training client_2...
   steps=20 | loss=0.4725 | epsilon=2.4813
Training client_3...
   steps=17 | loss=0.4594 | epsilon=2.6842

Round 1 complete.
Validation: {'val_loss': 0.46448139891328427, 'macro_auroc': 0.42555299882912645, 'per_class_auroc': {'Enlarged Cardiomediastinum': 0.5576923076923077, 'Cardiomegaly': 0.6228571428571429, 'Lung Opacity': 0.2823529411764706, 'Lung Lesion': 0.25, 'Edema': 0.3764705882352941, 'Consolidation': 0.45777777777777784, 'Pneumonia': 0.4375, 'Atelectasis': 0.5428571428571428, 'Pneumothorax': 0.5076086956521739, 'Pleural Effusion': 0.5330578512396694, 'Pleural Other': None, 'Fracture': 0.4285714285714286, 'Support Devices': 0.10989010989010989}}
Saved artifacts: /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/round_verification


In [ ]:
# E4 — Step 10: Continue Federated DP Training (Rounds 2–5)
import copy
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.accountants import RDPAccountant
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# 1. Maintain out-of-place residual addition patch
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

# 2. Paths and Directories
E4_DIR = PROJECT_ROOT / "results" / "E4_dp_federated"
ROUND_VERIFY_DIR = E4_DIR / "round_verification"
CHECKPOINT_DIR = E4_DIR / "checkpoints"
METRICS_DIR = E4_DIR / "metrics"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# 3. Load Round 1 Seed Checkpoint & Verification Summary
r1_checkpoint_path = ROUND_VERIFY_DIR / "round_01.pt"
r1_summary_path = ROUND_VERIFY_DIR / "round_01_summary.json"

assert r1_checkpoint_path.exists(), f"Missing Round 1 checkpoint: {r1_checkpoint_path}"
assert r1_summary_path.exists(), f"Missing Round 1 summary: {r1_summary_path}"

r1_checkpoint = torch.load(r1_checkpoint_path, map_location="cpu", weights_only=False)
with open(r1_summary_path, "r") as f:
    r1_summary = json.load(f)

# Initialize persistent history starting from Round 1
history = [
    {
        "round": 1,
        "global_val_loss": r1_summary["validation"]["val_loss"],
        "global_macro_auroc": r1_summary["validation"]["macro_auroc"],
        "per_class_auroc": r1_summary["validation"].get("per_class_auroc", {}),
        "clients": r1_summary["clients"],
        "duration_sec": r1_summary.get("elapsed_seconds", 0.0),
        "aggregation": "sample_weighted_FedAvg",
        "secure_rng": False,
    }
]

# Save initial standardized checkpoint for Round 1 in main checkpoint dir
torch.save(
    {
        "round": 1,
        "model_state_dict": r1_checkpoint["model_state_dict"],
        "accountant_states": r1_checkpoint["accountant_states"],
        "summary": history[0],
    },
    CHECKPOINT_DIR / "dp_federated_round_01.pt",
)

# 4. Global Model Setup & Parameter Restoration
global_model = MultimodalClassifier(num_classes=13)
global_model = ModuleValidator.fix(global_model)
for module in global_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

global_model.load_state_dict(r1_checkpoint["model_state_dict"], strict=True)
global_model = global_model.to(device)

# Load accountant states from Round 1
client_accountants_state = copy.deepcopy(r1_checkpoint["accountant_states"])

# Configuration constants
START_ROUND = 2
END_ROUND = 5
LOCAL_EPOCHS = 1
LEARNING_RATE = 1e-4
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
DELTA = 1e-5

print(f"=== Resuming E4 Federated DP Training: Rounds {START_ROUND}–{END_ROUND} ===")
print(f"Resumed from: {r1_checkpoint_path.name}")
print(f"Round 1 Val Loss: {history[0]['global_val_loss']:.4f} | Macro AUROC: {history[0]['global_macro_auroc']}\n")

# 5. Multi-Round Loop (Rounds 2–5)
for round_num in range(START_ROUND, END_ROUND + 1):
    round_start = time.time()
    print(f"\n--- Starting Round {round_num:02d}/{END_ROUND:02d} ---")

    global_state = {
        k: v.detach().cpu().clone()
        for k, v in global_model.state_dict().items()
    }

    client_states = []
    client_sizes = []
    client_summaries = []
    updated_accountant_states = {}

    for client_id, base_loader in client_loaders.items():
        print(f"  Training {client_id}...")

        local_model = MultimodalClassifier(num_classes=13)
        local_model = ModuleValidator.fix(local_model)
        for module in local_model.modules():
            if isinstance(module, nn.ReLU):
                module.inplace = False

        local_model.load_state_dict(global_state, strict=True)
        local_model = local_model.to(device)

        optimizer = torch.optim.AdamW(
            local_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
        )

        # Restore cumulative RDP accountant state for this client
        privacy_engine = PrivacyEngine(accountant="rdp")
        if client_id in client_accountants_state:
            privacy_engine.accountant.load_state_dict(client_accountants_state[client_id])

        private_model, private_optimizer, private_loader = (
            privacy_engine.make_private(
                module=local_model,
                optimizer=optimizer,
                data_loader=base_loader,
                noise_multiplier=NOISE_MULTIPLIER,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=True,
                loss_reduction="mean",
            )
        )

        private_model.train()
        loss_sum = 0.0
        steps = 0

        for _ in range(LOCAL_EPOCHS):
            for batch in private_loader:
                if isinstance(batch, dict):
                    images = batch["image"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True).float()
                    masks = batch["label_mask"].to(device, non_blocking=True).float()
                    sids = batch["sample_id"]
                    if torch.is_tensor(sids) or isinstance(sids, np.ndarray):
                        sids = sids.tolist()
                    sids = [int(x) for x in sids]
                else:
                    images, labels, masks, sids, _ = batch
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True).float()
                    masks = masks.to(device, non_blocking=True).float()
                    sids = [int(x) for x in sids]

                texts = torch.stack(
                    [embedding_by_id[sid] for sid in sids]
                ).to(device, non_blocking=True)

                private_optimizer.zero_grad(set_to_none=True)
                logits = private_model(images, texts)
                loss = masked_bce_with_logits(logits, labels, masks)
                loss.backward()
                private_optimizer.step()

                loss_sum += float(loss.item())
                steps += 1

        cumulative_epsilon = privacy_engine.get_epsilon(delta=DELTA)
        updated_accountant_states[client_id] = privacy_engine.accountant.state_dict()

        underlying = (
            private_model._module
            if hasattr(private_model, "_module")
            else private_model
        )
        client_states.append({
            k: v.detach().cpu().clone()
            for k, v in underlying.state_dict().items()
        })
        client_sizes.append(len(base_loader.dataset))

        mean_loss = loss_sum / max(steps, 1)
        client_summaries.append({
            "client": client_id,
            "dataset_size": len(base_loader.dataset),
            "optimizer_steps": steps,
            "mean_batch_loss": mean_loss,
            "cumulative_epsilon": float(cumulative_epsilon),
            "delta": DELTA,
            "noise_multiplier": NOISE_MULTIPLIER,
            "max_grad_norm": MAX_GRAD_NORM,
            "accountant": "RDPAccountant",
        })

        print(
            f"    [{client_id}] Steps={steps:2d} | "
            f"Loss={mean_loss:.4f} | "
            f"Cumulative ε={cumulative_epsilon:.4f}"
        )

        del private_model, private_optimizer, privacy_engine, local_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 6. Sample-Weighted FedAvg Aggregation
    total_size = sum(client_sizes)
    aggregated_state = {}

    for key in client_states[0]:
        tensors = [state[key] for state in client_states]
        if torch.is_floating_point(tensors[0]):
            aggregated_state[key] = sum(
                tensor.float() * (size / total_size)
                for tensor, size in zip(tensors, client_sizes)
            ).to(tensors[0].dtype)
        else:
            aggregated_state[key] = tensors[0].clone()

    global_model.load_state_dict(aggregated_state, strict=True)
    global_model.eval()

    # 7. Global Central Validation
    eval_res = evaluate_global(global_model, val_loader)
    val_loss = float(eval_res[0])
    val_macro_auroc = float(eval_res[1]) if eval_res[1] is not None else None
    per_class_auroc = eval_res[2] if len(eval_res) > 2 else {}

    validation = {
        "val_loss": val_loss,
        "macro_auroc": val_macro_auroc,
        "per_class_auroc": per_class_auroc,
    }

    record = {
        "round": round_num,
        "global_val_loss": validation["val_loss"],
        "global_macro_auroc": validation["macro_auroc"],
        "per_class_auroc": validation.get("per_class_auroc", {}),
        "clients": client_summaries,
        "duration_sec": time.time() - round_start,
        "aggregation": "sample_weighted_FedAvg",
        "secure_rng": False,
    }
    history.append(record)

    # 8. State Propagation and Checkpoint Saving
    client_accountants_state = updated_accountant_states

    checkpoint = {
        "round": round_num,
        "model_state_dict": global_model.state_dict(),
        "accountant_states": client_accountants_state,
        "summary": record,
    }
    torch.save(
        checkpoint,
        CHECKPOINT_DIR / f"dp_federated_round_{round_num:02d}.pt",
    )

    with open(METRICS_DIR / "dp_federated_rounds.json", "w") as f:
        json.dump(history, f, indent=2)

    auroc_str = f"{val_macro_auroc:.4f}" if val_macro_auroc is not None else "N/A"
    print(
        f"Round {round_num:02d} complete | "
        f"Val loss={validation['val_loss']:.4f} | "
        f"Macro AUROC={auroc_str} | "
        f"Duration={record['duration_sec']:.1f}s"
    )

# Save best and final checkpoints based on global validation loss
best_round_record = min(history, key=lambda x: x["global_val_loss"])
best_round_num = best_round_record["round"]
best_checkpoint_data = torch.load(
    CHECKPOINT_DIR / f"dp_federated_round_{best_round_num:02d}.pt",
    map_location="cpu",
    weights_only=False,
)
torch.save(best_checkpoint_data["model_state_dict"], CHECKPOINT_DIR / "best.pt")
torch.save(global_model.state_dict(), CHECKPOINT_DIR / "last.pt")

print("\nE4 continuation complete.")
print("Rounds completed: 1–5")
print(f"Best Round: {best_round_num} (Val Loss: {best_round_record['global_val_loss']:.4f})")
print("Checkpoints saved in:", CHECKPOINT_DIR)
print("Metrics saved in:    ", METRICS_DIR / "dp_federated_rounds.json")

=== Resuming E4 Federated DP Training: Rounds 2–5 ===
Resumed from: round_01.pt
Round 1 Val Loss: 0.4645 | Macro AUROC: 0.42555299882912645


--- Starting Round 02/05 ---
  Training client_1...
    [client_1] Steps=17 | Loss=0.3767 | Cumulative ε=3.2325
  Training client_2...
    [client_2] Steps=20 | Loss=0.4471 | Cumulative ε=2.9702
  Training client_3...
    [client_3] Steps=17 | Loss=0.4592 | Cumulative ε=3.2325
Round 02 complete | Val loss=0.4653 | Macro AUROC=0.4245 | Duration=10.2s

--- Starting Round 03/05 ---
  Training client_1...
    [client_1] Steps=17 | Loss=0.4125 | Cumulative ε=3.6769
  Training client_2...
    [client_2] Steps=20 | Loss=0.4306 | Cumulative ε=3.3681
  Training client_3...
    [client_3] Steps=17 | Loss=0.4343 | Cumulative ε=3.6769
Round 03 complete | Val loss=0.4664 | Macro AUROC=0.4270 | Duration=10.0s

--- Starting Round 04/05 ---
  Training client_1...
    [client_1] Steps=17 | Loss=0.4356 | Cumulative ε=4.0664
  Training client_2...
    [client_2] St

In [ ]:

import torch
import torch.nn.functional as F

def federated_masked_bce(logits, targets, masks):
    """
    Per-image masked BCE.
    Each image contributes one normalized loss before batch averaging.
    Use with Opacus loss_reduction="mean".
    """
    masks = masks.float()
    targets = targets.float()

    # Prevent NaN or uncertain targets from contaminating masked loss.
    targets = torch.nan_to_num(targets, nan=0.0)
    targets = torch.where(
        masks > 0,
        targets,
        torch.zeros_like(targets)
    )

    element_loss = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )

    per_image_numerator = (element_loss * masks).sum(dim=1)
    per_image_denominator = masks.sum(dim=1).clamp_min(1.0)

    per_image_loss = per_image_numerator / per_image_denominator
    return per_image_loss.mean()


In [ ]:
# E4 validation: test corrected per-image masked BCE on a real batch
import torch
from opacus.validators import ModuleValidator

batch = next(iter(client_loaders["client_1"]))

if isinstance(batch, dict):
    images = batch["image"].to(device)
    labels = batch["labels"].to(device).float()
    masks = batch["label_mask"].to(device).float()
    sample_ids = batch["sample_id"]
    if torch.is_tensor(sample_ids) or hasattr(sample_ids, "numpy"):
        sample_ids = sample_ids.tolist()
    sample_ids = [int(x) for x in sample_ids]
else:
    images, labels, masks, sample_ids, patient_ids = batch
    images = images.to(device)
    labels = labels.to(device).float()
    masks = masks.to(device).float()
    sample_ids = [int(x) for x in sample_ids]

# Fresh model for this loss-only sanity check
test_model = MultimodalClassifier(num_classes=13)
test_model = ModuleValidator.fix(test_model).to(device)
test_model.train()

texts = get_batch_text(sample_ids).to(device)
logits = test_model(images, texts)
loss = federated_masked_bce(logits, labels, masks)

# Assertions
assert logits.shape == labels.shape == masks.shape, f"Shape mismatch: {logits.shape} vs {labels.shape}"
assert torch.isfinite(loss), f"Loss is not finite: {loss.item()}"
assert torch.all((masks == 0) | (masks == 1)), "Masks must be strictly binary {0, 1}"
assert torch.isfinite(labels).all(), "Labels contain NaN or Inf"

loss.backward()
has_grad = any(
    p.grad is not None and torch.isfinite(p.grad).all()
    for p in test_model.parameters()
)
assert has_grad, "No finite gradients found across test model parameters"

print("Loss:                  ", round(loss.item(), 6))
print("Logits shape:          ", tuple(logits.shape))
print("Valid labels in batch: ", int(masks.sum().item()))
print("Loss and gradient sanity checks: PASS")

Loss:                   0.607495
Logits shape:           (16, 13)
Valid labels in batch:  46
Loss and gradient sanity checks: PASS


In [ ]:
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock

# 1. Out-of-place residual patch to satisfy autograd view-hook rules
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

# 2. Model setup with disabled in-place activations
dp_test_model = MultimodalClassifier(num_classes=13)
dp_test_model = ModuleValidator.fix(dp_test_model)
for module in dp_test_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False
dp_test_model = dp_test_model.to(device)
dp_test_model.train()

dp_test_optimizer = torch.optim.AdamW(dp_test_model.parameters(), lr=1e-4)
dp_test_engine = PrivacyEngine(accountant="rdp")

dp_model, dp_optimizer, dp_loader = dp_test_engine.make_private(
    module=dp_test_model,
    optimizer=dp_test_optimizer,
    data_loader=client_loaders["client_1"],
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    poisson_sampling=True,
    loss_reduction="mean",
)

# 3. Fetch a batch handling dict or tuple formats
batch = next(iter(dp_loader))

if isinstance(batch, dict):
    images = batch["image"].to(device)
    labels = batch["labels"].to(device).float()
    masks = batch["label_mask"].to(device).float()
    sids = batch["sample_id"]
    if torch.is_tensor(sids) or hasattr(sids, "numpy"):
        sids = sids.tolist()
    sample_ids = [int(x) for x in sids]
else:
    images, labels, masks, sample_ids, patient_ids = batch
    images = images.to(device)
    labels = labels.to(device).float()
    masks = masks.to(device).float()
    sample_ids = [int(x) for x in sample_ids]

texts = get_batch_text(sample_ids).to(device)

# 4. Forward, Backward & Step under Opacus wrapper
dp_optimizer.zero_grad(set_to_none=True)
logits = dp_model(images, texts)
loss = federated_masked_bce(logits, labels, masks)
loss.backward()

# 5. Inspect per-sample gradient registration
sample_grad_found = False
for name, param in dp_model.named_parameters():
    if hasattr(param, "grad_sample") and param.grad_sample is not None:
        sample_grad_found = True
        batch_dim = param.grad_sample.shape[0]
        print(f"Per-sample gradient detected on layer '{name}' with batch dimension: {batch_dim}")
        break

dp_optimizer.step()

assert sample_grad_found, "Opacus did not populate `grad_sample` on model parameters!"
print("Loss value:                           ", round(loss.item(), 6))
print("Opacus per-sample gradient validation: PASS")

Per-sample gradient detected on layer '_module.image_encoder.conv1.weight' with batch dimension: 12
Loss value:                            0.672268
Opacus per-sample gradient validation: PASS


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

CORRECTED_E4_DIR = E4_DIR / "corrected_run"
CORRECTED_E4_CHECKPOINTS = CORRECTED_E4_DIR / "checkpoints"
CORRECTED_E4_METRICS = CORRECTED_E4_DIR / "metrics"
CORRECTED_E4_ACCOUNTANTS = CORRECTED_E4_DIR / "accountants"

for folder in [
    CORRECTED_E4_CHECKPOINTS,
    CORRECTED_E4_METRICS,
    CORRECTED_E4_ACCOUNTANTS,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Point starting_checkpoint to E2 multimodal weights for a clean 5-round run
e2_checkpoint_path = E2_DIR / "checkpoints" / "best.pt"
if not e2_checkpoint_path.exists():
    e2_checkpoint_path = E2_DIR / "checkpoints" / "best_model.pt"

config = {
    "experiment": "E4_corrected_DP_federated_multimodal",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "CheXpert Plus",
    "subset_size": 1000,
    "clients": 3,
    "rounds": 5,
    "local_epochs": 1,
    "aggregation": "sample_weighted_FedAvg",
    "noise_multiplier": 1.0,
    "max_grad_norm": 1.0,
    "delta": 1e-5,
    "accountant": "RDPAccountant",
    "loss": "per_image_masked_BCE",
    "secure_rng": False,
    "privacy_unit": "image-level experimental accounting; not patient-level DP",
    "starting_checkpoint": str(e2_checkpoint_path),
    "status": "setup_only_not_yet_trained",
}

with open(CORRECTED_E4_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Corrected E4 directory:", CORRECTED_E4_DIR)
print("Configuration saved to:", CORRECTED_E4_DIR / "config.json")
print("Original pilot files remain untouched.")
print("No training has been started.")

Corrected E4 directory: /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/corrected_run
Configuration saved to: /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/corrected_run/config.json
Original pilot files remain untouched.
No training has been started.


In [ ]:
import json
import torch

# 1. Confirm the E2 starting checkpoint exists
e2_checkpoint = E2_DIR / "checkpoints" / "best.pt"
if not e2_checkpoint.exists():
    e2_checkpoint = E2_DIR / "checkpoints" / "best_model.pt"

assert e2_checkpoint.exists(), f"Missing E2 checkpoint: {e2_checkpoint}"

# 2. Confirm the corrected experiment configuration
config_path = CORRECTED_E4_DIR / "config.json"
assert config_path.exists(), f"Missing config file: {config_path}"

with open(config_path, "r") as f:
    corrected_config = json.load(f)

assert corrected_config["loss"] == "per_image_masked_BCE", f"Unexpected loss: {corrected_config['loss']}"
assert corrected_config["rounds"] == 5, f"Unexpected rounds: {corrected_config['rounds']}"

# 3. Inspect checkpoint format without modifying it
e2_state = torch.load(e2_checkpoint, map_location="cpu", weights_only=False)

print("E2 checkpoint path:", e2_checkpoint)
print("Checkpoint type:   ", type(e2_state).__name__)

if isinstance(e2_state, dict):
    # Check if this is a raw state_dict or a container checkpoint dictionary
    is_state_dict = any("weight" in k or "bias" in k for k in e2_state.keys())
    print("Is direct state_dict:", is_state_dict)
    print("Sample top-level keys:", list(e2_state.keys())[:10])

print("\nCorrected E4 configuration verification: PASS")

E2 checkpoint path: /content/drive/MyDrive/DP-MMFL/results/E2_multimodal/checkpoints/best.pt
Checkpoint type:    OrderedDict
Is direct state_dict: True
Sample top-level keys: ['image_encoder.conv1.weight', 'image_encoder.bn1.weight', 'image_encoder.bn1.bias', 'image_encoder.bn1.running_mean', 'image_encoder.bn1.running_var', 'image_encoder.bn1.num_batches_tracked', 'image_encoder.layer1.0.conv1.weight', 'image_encoder.layer1.0.bn1.weight', 'image_encoder.layer1.0.bn1.bias', 'image_encoder.layer1.0.bn1.running_mean']

Corrected E4 configuration verification: PASS


In [ ]:
# Step 11 — Initialize Corrected E4: Architecture Adaptation & Fresh Accountants
import json
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.accountants import RDPAccountant
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock

# 1. Maintain out-of-place residual addition patch for autograd backward hooks
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

# 2. Load E2 multimodal checkpoint
e2_checkpoint = E2_DIR / "checkpoints" / "best.pt"
if not e2_checkpoint.exists():
    e2_checkpoint = E2_DIR / "checkpoints" / "best_model.pt"

e2_state = torch.load(e2_checkpoint, map_location="cpu", weights_only=False)

# 3. Instantiate base model, convert BatchNorm -> GroupNorm, and disable in-place ReLUs
corrected_global_model = MultimodalClassifier(num_classes=13)
corrected_global_model = ModuleValidator.fix(corrected_global_model)

for module in corrected_global_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

# 4. Transfer learned parameters (strict=False accommodates BN -> GN conversion)
load_result = corrected_global_model.load_state_dict(e2_state, strict=False)

print("Missing keys:   ", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))

# Verify discarded keys are exclusively BatchNorm tracking buffers
unexpected_learned = [
    key for key in load_result.unexpected_keys
    if not any(marker in key for marker in ("running_mean", "running_var", "num_batches_tracked"))
]
assert not unexpected_learned, f"Unexpected learned E2 parameters found: {unexpected_learned[:10]}"
print("Parameter transfer validation: PASS (Only standard BatchNorm buffers omitted)")

corrected_global_model = corrected_global_model.to(device)

# 5. Pre-training baseline central evaluation
eval_res = evaluate_global(corrected_global_model, val_loader)
init_val_loss = float(eval_res[0])
init_val_auroc = float(eval_res[1]) if eval_res[1] is not None else None
auroc_str = f"{init_val_auroc:.4f}" if init_val_auroc is not None else "N/A"

print(f"\n=== Corrected E4 Pre-Training Central Evaluation (Round 00) ===")
print(f"Initial Val Loss:   {init_val_loss:.4f}")
print(f"Initial Macro AUROC:{auroc_str}")

# 6. Initialize fresh zero-state RDP accountants for all 3 federated clients
fresh_accountant_states = {}
for client_id in client_loaders.keys():
    acc = RDPAccountant()
    fresh_accountant_states[client_id] = acc.state_dict()

# 7. Persist clean Round 00 artifacts in corrected_run/
init_summary = {
    "round": 0,
    "val_loss": init_val_loss,
    "macro_auroc": init_val_auroc,
    "per_class_auroc": eval_res[2] if len(eval_res) > 2 else {},
    "note": "Fresh initialization from E2 weights prior to DP federated training",
}

torch.save(
    {
        "round": 0,
        "model_state_dict": corrected_global_model.state_dict(),
        "accountant_states": fresh_accountant_states,
        "summary": init_summary,
    },
    CORRECTED_E4_CHECKPOINTS / "dp_federated_round_00.pt",
)

with open(CORRECTED_E4_ACCOUNTANTS / "accountant_states_round_00.json", "w") as f:
    json.dump(fresh_accountant_states, f, indent=2)

with open(CORRECTED_E4_METRICS / "init_metrics.json", "w") as f:
    json.dump(init_summary, f, indent=2)

print("\nRound 00 checkpoint, metrics, and fresh accountant states saved to:")
print(f" - {CORRECTED_E4_CHECKPOINTS / 'dp_federated_round_00.pt'}")
print(f" - {CORRECTED_E4_ACCOUNTANTS / 'accountant_states_round_00.json'}")

Missing keys:    0
Unexpected keys: 60
Parameter transfer validation: PASS (Only standard BatchNorm buffers omitted)

=== Corrected E4 Pre-Training Central Evaluation (Round 00) ===
Initial Val Loss:   0.4634
Initial Macro AUROC:0.4244

Round 00 checkpoint, metrics, and fresh accountant states saved to:
 - /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/corrected_run/checkpoints/dp_federated_round_00.pt
 - /content/drive/MyDrive/DP-MMFL/results/E4_dp_federated/corrected_run/accountants/accountant_states_round_00.json


In [ ]:
# ==============================================================================
# E4 Corrected: Multi-Round Federated DP-SGD Loop (Rounds 1–5)
# Starts from Round 00 checkpoint, persists RDP accountants, saves per-round artifacts
# ==============================================================================
import copy
import json
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.accountants import RDPAccountant
from opacus.validators import ModuleValidator
from torchvision.models.resnet import BasicBlock

# 1. Out-of-place residual shortcut patch to satisfy autograd backward hooks
def safe_basicblock_forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    if self.downsample is not None:
        identity = self.downsample(x)
    out = out + identity
    out = self.relu(out)
    return out

BasicBlock.forward = safe_basicblock_forward

# 2. Paths and Verification
CORRECTED_E4_DIR = E4_DIR / "corrected_run"
CHECKPOINTS_DIR = CORRECTED_E4_DIR / "checkpoints"
METRICS_DIR = CORRECTED_E4_DIR / "metrics"
ACCOUNTANTS_DIR = CORRECTED_E4_DIR / "accountants"

r0_ckpt_path = CHECKPOINTS_DIR / "dp_federated_round_00.pt"
assert r0_ckpt_path.exists(), f"Missing Round 00 checkpoint at: {r0_ckpt_path}"

# 3. Model Initializer Factory
def create_clean_dp_model():
    model = MultimodalClassifier(num_classes=13)
    model = ModuleValidator.fix(model)
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
    return model

# 4. Load Global Model State from Round 00
global_model = create_clean_dp_model()
r0_checkpoint = torch.load(r0_ckpt_path, map_location="cpu", weights_only=False)
global_model.load_state_dict(r0_checkpoint["model_state_dict"], strict=True)
global_model = global_model.to(device)

# Load zero-state accountants initialized in Step 11
client_accountants_state = copy.deepcopy(r0_checkpoint["accountant_states"])

# 5. Hyperparameters
TOTAL_ROUNDS = 5
LOCAL_EPOCHS = 1
LEARNING_RATE = 1e-4
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
DELTA = 1e-5

client_sizes = {cid: len(loader.dataset) for cid, loader in client_loaders.items()}
total_dataset_size = sum(client_sizes.values())

training_history = []
best_val_loss = float("inf")
best_round = -1

print("=== Starting E4 Corrected 5-Round DP-FedAvg Training ===")
print(f"Clients: {list(client_loaders.keys())} | Total Dataset: {total_dataset_size} samples")
print(f"Privacy Config: σ={NOISE_MULTIPLIER}, C={MAX_GRAD_NORM}, δ={DELTA}, Loss=per_image_masked_bce\n")

# 6. Main Federated Rounds Loop (1 through 5)
for round_idx in range(1, TOTAL_ROUNDS + 1):
    round_start = time.time()
    print(f"=== Starting Round {round_idx:02d}/{TOTAL_ROUNDS:02d} ===")

    global_weights = {
        k: v.detach().cpu().clone()
        for k, v in global_model.state_dict().items()
    }

    client_states = []
    client_summaries = []
    updated_accountant_states = {}

    for client_id, base_loader in client_loaders.items():
        # Instantiate fresh local model and synchronize with broadcast server parameters
        local_model = create_clean_dp_model()
        local_model.load_state_dict(global_weights, strict=True)
        local_model = local_model.to(device)

        optimizer = torch.optim.AdamW(
            local_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
        )

        # Initialize engine and restore historical RDP accounting state
        privacy_engine = PrivacyEngine(accountant="rdp")
        if client_id in client_accountants_state and client_accountants_state[client_id] is not None:
            privacy_engine.accountant.load_state_dict(client_accountants_state[client_id])

        private_model, private_optimizer, private_loader = (
            privacy_engine.make_private(
                module=local_model,
                optimizer=optimizer,
                data_loader=base_loader,
                noise_multiplier=NOISE_MULTIPLIER,
                max_grad_norm=MAX_GRAD_NORM,
                poisson_sampling=True,
                loss_reduction="mean",
            )
        )

        private_model.train()
        loss_accum = 0.0
        steps = 0

        for _ in range(LOCAL_EPOCHS):
            for batch in private_loader:
                if isinstance(batch, dict):
                    images = batch["image"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True).float()
                    masks = batch["label_mask"].to(device, non_blocking=True).float()
                    sids = batch["sample_id"]
                    if torch.is_tensor(sids) or hasattr(sids, "numpy"):
                        sids = sids.tolist()
                    sids = [int(x) for x in sids]
                else:
                    images, labels, masks, sids, _ = batch
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True).float()
                    masks = masks.to(device, non_blocking=True).float()
                    sids = [int(x) for x in sids]

                texts = get_batch_text(sids).to(device, non_blocking=True)

                private_optimizer.zero_grad(set_to_none=True)
                logits = private_model(images, texts)
                loss = federated_masked_bce(logits, labels, masks)
                loss.backward()
                private_optimizer.step()

                loss_accum += float(loss.item())
                steps += 1

        cumulative_eps = privacy_engine.get_epsilon(delta=DELTA)
        updated_accountant_states[client_id] = privacy_engine.accountant.state_dict()

        # Capture unwrapped state dictionary
        underlying = (
            private_model._module
            if hasattr(private_model, "_module")
            else private_model
        )
        client_states.append({
            k: v.detach().cpu().clone()
            for k, v in underlying.state_dict().items()
        })

        mean_loss = loss_accum / max(steps, 1)
        client_summaries.append({
            "client": client_id,
            "dataset_size": client_sizes[client_id],
            "optimizer_steps": steps,
            "mean_loss": mean_loss,
            "cumulative_epsilon": float(cumulative_eps),
            "delta": DELTA,
        })

        print(
            f"  [{client_id}] Steps: {steps:2d} | "
            f"Local Loss: {mean_loss:.4f} | "
            f"Cumulative ε: {cumulative_eps:.4f}"
        )

        del private_model, private_optimizer, privacy_engine, local_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 7. Sample-Weighted FedAvg Aggregation
    aggregated_weights = {}
    for key in client_states[0]:
        tensors = [state[key] for state in client_states]
        if torch.is_floating_point(tensors[0]):
            aggregated_weights[key] = sum(
                tensor.float() * (client_sizes[cid] / total_dataset_size)
                for tensor, cid in zip(tensors, client_loaders.keys())
            ).to(tensors[0].dtype)
        else:
            aggregated_weights[key] = tensors[0].clone()

    global_model.load_state_dict(aggregated_weights, strict=True)
    global_model.eval()

    # 8. Central Evaluation on Validation Split
    val_out = evaluate_global(global_model, val_loader)
    val_loss = float(val_out[0])
    val_macro_auroc = float(val_out[1]) if val_out[1] is not None else None
    per_class = val_out[2] if len(val_out) > 2 else {}
    round_duration = time.time() - round_start

    auroc_str = f"{val_macro_auroc:.4f}" if val_macro_auroc is not None else "N/A"
    print(
        f"Round {round_idx:02d} Complete ({round_duration:.1f}s) | "
        f"Global Val Loss: {val_loss:.4f} | "
        f"Macro AUROC: {auroc_str}\n"
    )

    round_record = {
        "round": round_idx,
        "global_val_loss": val_loss,
        "global_macro_auroc": val_macro_auroc,
        "per_class_auroc": per_class,
        "clients": client_summaries,
        "duration_sec": round_duration,
        "loss_type": "per_image_masked_bce",
        "aggregation": "sample_weighted_FedAvg",
        "secure_rng": False,
    }
    training_history.append(round_record)

    # Propagate cumulative accounting states to the next round
    client_accountants_state = updated_accountant_states

    # 9. Artifact Persistence per Round
    round_checkpoint = {
        "round": round_idx,
        "model_state_dict": global_model.state_dict(),
        "accountant_states": client_accountants_state,
        "summary": round_record,
    }
    torch.save(round_checkpoint, CHECKPOINTS_DIR / f"dp_federated_round_{round_idx:02d}.pt")

    with open(METRICS_DIR / "dp_federated_rounds.json", "w") as f:
        json.dump(training_history, f, indent=2)

    with open(ACCOUNTANTS_DIR / f"accountant_states_round_{round_idx:02d}.json", "w") as f:
        json.dump(client_accountants_state, f, indent=2)

    # Track best model based on minimum validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_round = round_idx
        torch.save(global_model.state_dict(), CHECKPOINTS_DIR / "best.pt")

torch.save(global_model.state_dict(), CHECKPOINTS_DIR / "last.pt")

print("=== E4 Corrected 5-Round Run Complete ===")
print(f"Optimal Round:        {best_round} (Val Loss: {best_val_loss:.4f})")
print(f"Final Weights Saved:  {CHECKPOINTS_DIR / 'last.pt'}")
print(f"Best Weights Saved:   {CHECKPOINTS_DIR / 'best.pt'}")
print(f"Metrics Log Saved:    {METRICS_DIR / 'dp_federated_rounds.json'}")

=== Starting E4 Corrected 5-Round DP-FedAvg Training ===
Clients: ['client_1', 'client_2', 'client_3'] | Total Dataset: 835 samples
Privacy Config: σ=1.0, C=1.0, δ=1e-05, Loss=per_image_masked_bce

=== Starting Round 01/05 ===
  [client_1] Steps: 17 | Local Loss: 0.3681 | Cumulative ε: 2.6842
  [client_2] Steps: 20 | Local Loss: 0.4137 | Cumulative ε: 2.4813
  [client_3] Steps: 17 | Local Loss: 0.4247 | Cumulative ε: 2.6842
Round 01 Complete (14.4s) | Global Val Loss: 0.4650 | Macro AUROC: 0.4289

=== Starting Round 02/05 ===
  [client_1] Steps: 17 | Local Loss: 0.3671 | Cumulative ε: 3.2325
  [client_2] Steps: 20 | Local Loss: 0.4317 | Cumulative ε: 2.9702
  [client_3] Steps: 17 | Local Loss: 0.4760 | Cumulative ε: 3.2325
Round 02 Complete (10.3s) | Global Val Loss: 0.4660 | Macro AUROC: 0.4300

=== Starting Round 03/05 ===
  [client_1] Steps: 17 | Local Loss: 0.3620 | Cumulative ε: 3.6769
  [client_2] Steps: 20 | Local Loss: 0.4124 | Cumulative ε: 3.3681
  [client_3] Steps: 17 | Loca

In [ ]:
import json
import torch

corrected_metrics_path = CORRECTED_E4_METRICS / "dp_federated_rounds.json"
assert corrected_metrics_path.exists(), f"Metrics file missing: {corrected_metrics_path}"

with open(corrected_metrics_path, "r") as f:
    corrected_history = json.load(f)

print("Metric records:", len(corrected_history))
for record in corrected_history:
    print(
        f"Round {record['round']}: "
        f"val_loss={record['global_val_loss']:.4f}, "
        f"macro_AUROC={record['global_macro_auroc']:.4f}"
    )

for name in ["best.pt", "last.pt"]:
    path = CORRECTED_E4_CHECKPOINTS / name
    assert path.exists(), f"Missing checkpoint: {path}"
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    print(f"{name}: readable ({type(checkpoint).__name__})")

print("Corrected E4 artifact verification: PASS")

Metric records: 5
Round 1: val_loss=0.4650, macro_AUROC=0.4289
Round 2: val_loss=0.4660, macro_AUROC=0.4300
Round 3: val_loss=0.4677, macro_AUROC=0.4438
Round 4: val_loss=0.4689, macro_AUROC=0.4453
Round 5: val_loss=0.4702, macro_AUROC=0.4419
best.pt: readable (OrderedDict)
last.pt: readable (OrderedDict)
Corrected E4 artifact verification: PASS


In [ ]:
# ==============================================================================
# Milestone: Generate Single Machine-Readable Summary (master_summary.json)
# Aggregates E1–E4 metrics, checkpoint metadata, and experimental parameters
# ==============================================================================
import json
import time
from datetime import datetime, timezone
from pathlib import Path
import torch

REPORT_DIR = PROJECT_ROOT / "results" / "consolidated_evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# File sources
p_e1 = PROJECT_ROOT / "results" / "E1_image_only" / "metrics" / "best_metrics.json"
p_e2 = PROJECT_ROOT / "results" / "E2_multimodal" / "metrics" / "best_metrics.json"
p_e3 = PROJECT_ROOT / "results" / "E3_federated" / "metrics" / "training_history.json"
p_e4_corr_metrics = CORRECTED_E4_METRICS / "dp_federated_rounds.json"
p_e4_corr_ckpt = CORRECTED_E4_CHECKPOINTS / "best.pt"
p_e4_corr_last = CORRECTED_E4_CHECKPOINTS / "last.pt"

# Verify all required paths exist
for path, name in [
    (p_e1, "E1 metrics"),
    (p_e2, "E2 metrics"),
    (p_e3, "E3 metrics"),
    (p_e4_corr_metrics, "E4 corrected metrics"),
    (p_e4_corr_ckpt, "E4 corrected best.pt"),
    (p_e4_corr_last, "E4 corrected last.pt"),
]:
    assert path.exists(), f"Missing required file: {name} at {path}"

# 1. Parse E1 & E2
with open(p_e1) as f:
    d_e1 = json.load(f)
with open(p_e2) as f:
    d_e2 = json.load(f)

# 2. Parse E3 (extract peak AUROC and min val loss rounds)
with open(p_e3) as f:
    d_e3 = json.load(f)
best_auroc_e3 = max(d_e3, key=lambda x: x["global_val_macro_auroc"])
best_loss_e3 = min(d_e3, key=lambda x: x["global_val_loss"])

# 3. Parse E4 Corrected
with open(p_e4_corr_metrics) as f:
    d_e4 = json.load(f)
best_auroc_e4 = max(d_e4, key=lambda x: x["global_macro_auroc"])
best_loss_e4 = min(d_e4, key=lambda x: x["global_val_loss"])

# 4. Checkpoint inspection metadata
best_ckpt_stat = p_e4_corr_ckpt.stat()
last_ckpt_stat = p_e4_corr_last.stat()

master_summary = {
    "metadata": {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "CheXpert Plus (pilot subset)",
        "total_subset_size": 1000,
        "split_strategy": "Patient-disjoint train/val",
        "num_classes": 13,
        "labels": TARGET_COLUMNS,
        "notes": (
            "Pilot evaluation across 1,000 images. Does not represent the full "
            "CheXpert Plus corpus. Models differ in normalization (BatchNorm vs GroupNorm) "
            "and loss reduction."
        ),
    },
    "experiments": {
        "E1": {
            "name": "Centralized Image-Only",
            "modality": "Vision (ResNet18)",
            "normalization": "BatchNorm2d",
            "loss_reduction": "Batch-masked BCE",
            "privacy": "None",
            "best_macro_auroc": {
                "value": 0.5392,
                "epoch": 3,
            },
            "best_val_loss": {
                "value": 0.4956,
                "epoch": 3,
            },
            "checkpoint_path": "results/E1_image_only/checkpoints/best.pt",
        },
        "E2": {
            "name": "Centralized Multimodal",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "BatchNorm2d",
            "loss_reduction": "Batch-masked BCE",
            "privacy": "None",
            "best_macro_auroc": {
                "value": 0.5209,
                "epoch": 3,
            },
            "best_val_loss": {
                "value": 0.4720,
                "epoch": 2,
            },
            "checkpoint_path": "results/E2_multimodal/checkpoints/best.pt",
        },
        "E3": {
            "name": "Federated Multimodal (FedAvg)",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "BatchNorm2d",
            "clients": 3,
            "rounds": 5,
            "aggregation": "Sample-weighted FedAvg",
            "privacy": "None",
            "best_macro_auroc": {
                "value": 0.5757,
                "round": 4,
            },
            "best_val_loss": {
                "value": 0.5005,
                "round": 3,
            },
            "checkpoint_path": "results/E3_federated/checkpoints/best.pt",
        },
        "E4_corrected": {
            "name": "DP Federated Multimodal (DP-FedAvg)",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "GroupNorm (ModuleValidator.fix)",
            "clients": 3,
            "rounds": 5,
            "aggregation": "Sample-weighted FedAvg",
            "loss_reduction": "Per-image masked BCE",
            "privacy_parameters": {
                "accountant": "RDPAccountant",
                "target_delta": 1e-5,
                "noise_multiplier": 1.0,
                "max_grad_norm": 1.0,
                "poisson_sampling": True,
                "secure_rng": False,
                "privacy_unit": "image-level",
                "final_epsilons": {
                    "client_1": 4.4203,
                    "client_2": 4.0383,
                    "client_3": 4.4203,
                },
            },
            "round_00_baseline": {
                "val_loss": 0.4634,
                "macro_auroc": 0.4244,
            },
            "best_macro_auroc": {
                "value": 0.4453,
                "round": 4,
            },
            "best_val_loss": {
                "value": 0.4650,
                "round": 1,
            },
            "round_5_final": {
                "val_loss": 0.4702,
                "macro_auroc": 0.4419,
            },
            "checkpoint_paths": {
                "best": "results/E4_dp_federated/corrected_run/checkpoints/best.pt",
                "last": "results/E4_dp_federated/corrected_run/checkpoints/last.pt",
            },
        },
    },
    "experimental_caveats": [
        "1,000-image pilot subset: Results serve as pipeline verification, not benchmark clinical claims.",
        "Normalization change: E1–E3 use BatchNorm; E4 uses GroupNorm to enable per-sample gradient tracking.",
        "Loss formulation: E1–E3 use batch-normalized valid-mask loss; E4 uses per-image masked BCE for DP-SGD compliance.",
        "Privacy bounds: Image-level sampling, not patient-level DP; PRNG mode is experimental (torchcsprng unavailable).",
        "Dual criteria divergence: Best AUROC and minimum validation loss select different rounds across all federated runs.",
    ],
}

master_path = REPORT_DIR / "master_summary.json"
with open(master_path, "w") as f:
    json.dump(master_summary, f, indent=2)

print(f"Master summary generated: {master_path}")
print(f"Summary keys: {list(master_summary.keys())}")
print(f"File size: {master_path.stat().st_size / 1024:.2f} KB")

AssertionError: Missing required file: E1 metrics at /content/drive/MyDrive/DP-MMFL/results/E1_image_only/metrics/best_metrics.json

In [ ]:
from pathlib import Path

for exp in ["E1_image_only", "E2_multimodal", "E3_federated", "E4_dp_federated"]:
    exp_dir = PROJECT_ROOT / "results" / exp
    print(f"\n[{exp}]")
    if exp_dir.exists():
        for p in sorted(exp_dir.rglob("*.json")):
            print(f"  metrics: {p.relative_to(PROJECT_ROOT)}")
        for p in sorted(exp_dir.rglob("*.pt")):
            print(f"  weights: {p.relative_to(PROJECT_ROOT)}")
    else:
        print("  Directory not found!")


[E1_image_only]
  Directory not found!

[E2_multimodal]
  metrics: results/E2_multimodal/metrics/e2_training_history.json
  metrics: results/E2_multimodal/metrics/training_history.json
  metrics: results/E2_multimodal/split_info.json
  weights: results/E2_multimodal/checkpoints/best.pt
  weights: results/E2_multimodal/checkpoints/best_model.pt
  weights: results/E2_multimodal/checkpoints/epoch_01.pt
  weights: results/E2_multimodal/checkpoints/epoch_02.pt
  weights: results/E2_multimodal/checkpoints/epoch_03.pt
  weights: results/E2_multimodal/checkpoints/last.pt
  weights: results/E2_multimodal/text_embeddings.pt

[E3_federated]
  metrics: results/E3_federated/metrics/client_partitions.json
  metrics: results/E3_federated/metrics/training_history.json
  weights: results/E3_federated/checkpoints/best.pt
  weights: results/E3_federated/checkpoints/last.pt
  weights: results/E3_federated/checkpoints/round_01.pt
  weights: results/E3_federated/checkpoints/round_02.pt
  weights: results/E

In [ ]:
# ==============================================================================
# Milestone: Generate Machine-Readable Summary (master_summary.json)
# Maps verified file paths and records E1–E4 comparative findings
# ==============================================================================
import json
from datetime import datetime, timezone
from pathlib import Path

REPORT_DIR = PROJECT_ROOT / "results" / "consolidated_evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load E2 Metrics
p_e2 = PROJECT_ROOT / "results" / "E2_multimodal" / "metrics" / "training_history.json"
with open(p_e2, "r") as f:
    d_e2 = json.load(f)

# 2. Load E3 Metrics
p_e3 = PROJECT_ROOT / "results" / "E3_federated" / "metrics" / "training_history.json"
with open(p_e3, "r") as f:
    d_e3 = json.load(f)

# 3. Load E4 Corrected Metrics
p_e4 = CORRECTED_E4_METRICS / "dp_federated_rounds.json"
with open(p_e4, "r") as f:
    d_e4 = json.load(f)

# Optional: Load E4 Pilot Metrics for comparison
p_e4_pilot = PROJECT_ROOT / "results" / "E4_dp_federated" / "metrics" / "dp_federated_rounds.json"
d_e4_pilot = None
if p_e4_pilot.exists():
    with open(p_e4_pilot, "r") as f:
        d_e4_pilot = json.load(f)

master_summary = {
    "metadata": {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "CheXpert Plus (pilot subset)",
        "total_subset_size": 1000,
        "split_strategy": "Patient-disjoint train/val (165 val images)",
        "num_classes": 13,
        "labels": TARGET_COLUMNS if "TARGET_COLUMNS" in globals() else 13,
        "notes": (
            "Pilot evaluation across 1,000 images. Does not represent the full "
            "CheXpert Plus corpus. Models differ in normalization (BatchNorm vs GroupNorm) "
            "and loss reduction."
        ),
    },
    "experiments": {
        "E1": {
            "name": "Centralized Image-Only Baseline",
            "modality": "Vision (ResNet18)",
            "normalization": "BatchNorm2d",
            "loss_reduction": "Batch-masked BCE",
            "privacy": "None",
            "best_macro_auroc": {"value": 0.5392, "epoch": 3},
            "best_val_loss": {"value": 0.4956, "epoch": 3},
            "artifact_status": "Reference pilot metrics (directory unpersisted)",
        },
        "E2": {
            "name": "Centralized Multimodal Baseline",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "BatchNorm2d",
            "loss_reduction": "Batch-masked BCE",
            "privacy": "None",
            "best_macro_auroc": {"value": 0.5209, "epoch": 3},
            "best_val_loss": {"value": 0.4720, "epoch": 2},
            "metrics_path": "results/E2_multimodal/metrics/training_history.json",
            "checkpoint_path": "results/E2_multimodal/checkpoints/best.pt",
        },
        "E3": {
            "name": "Federated Multimodal (FedAvg)",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "BatchNorm2d",
            "clients": 3,
            "rounds": 5,
            "aggregation": "Sample-weighted FedAvg",
            "privacy": "None",
            "best_macro_auroc": {"value": 0.5757, "round": 4},
            "best_val_loss": {"value": 0.5005, "round": 3},
            "metrics_path": "results/E3_federated/metrics/training_history.json",
            "checkpoint_path": "results/E3_federated/checkpoints/best.pt",
        },
        "E4_pilot": {
            "name": "DP Federated Multimodal (Pilot Loss)",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "GroupNorm",
            "clients": 3,
            "rounds": 5,
            "loss_reduction": "Batch-level label normalization",
            "privacy_parameters": {
                "accountant": "RDPAccountant",
                "delta": 1e-5,
                "noise_multiplier": 1.0,
                "max_grad_norm": 1.0,
                "secure_rng": False,
                "final_epsilons": {"client_1": 4.4203, "client_2": 4.0383, "client_3": 4.4203},
            },
            "best_macro_auroc": {"value": 0.4384, "round": 5},
            "best_val_loss": {"value": 0.4645, "round": 1},
            "checkpoint_path": "results/E4_dp_federated/checkpoints/best.pt",
        },
        "E4_corrected": {
            "name": "DP Federated Multimodal (Per-Image Masked BCE)",
            "modality": "Vision + Text (BioClinicalBERT)",
            "normalization": "GroupNorm (ModuleValidator.fix)",
            "clients": 3,
            "rounds": 5,
            "loss_reduction": "Per-image masked BCE (Opacus compliant)",
            "privacy_parameters": {
                "accountant": "RDPAccountant",
                "delta": 1e-5,
                "noise_multiplier": 1.0,
                "max_grad_norm": 1.0,
                "poisson_sampling": True,
                "secure_rng": False,
                "privacy_unit": "image-level",
                "final_epsilons": {"client_1": 4.4203, "client_2": 4.0383, "client_3": 4.4203},
            },
            "round_00_baseline": {"val_loss": 0.4634, "macro_auroc": 0.4244},
            "best_macro_auroc": {"value": 0.4453, "round": 4},
            "best_val_loss": {"value": 0.4650, "round": 1},
            "round_05_final": {"val_loss": 0.4702, "macro_auroc": 0.4419},
            "checkpoint_paths": {
                "best": "results/E4_dp_federated/corrected_run/checkpoints/best.pt",
                "last": "results/E4_dp_federated/corrected_run/checkpoints/last.pt",
            },
        },
    },
    "experimental_caveats": [
        "1,000-image pilot subset: Confines claims to pipeline feasibility and comparative trend observation rather than definitive clinical performance.",
        "Normalization change: E1–E3 utilize standard BatchNorm2d; E4 converts to GroupNorm via ModuleValidator to support per-sample gradient tracking.",
        "Loss formulation: E4 Corrected computes scalar per-sample losses before mean reduction, ensuring mathematically sound gradient clipping under Opacus.",
        "Privacy boundaries: Guarantees represent image-level Poisson sampling, not patient-level DP; PRNG mode is experimental (torchcsprng unavailable).",
        "Dual-metric divergence: Minimum validation loss and maximum macro AUROC identify different optimal checkpoints across all federated runs.",
    ],
}

master_path = REPORT_DIR / "master_summary.json"
with open(master_path, "w") as f:
    json.dump(master_summary, f, indent=2)

print(f"Master summary saved to: {master_path}")
print(f"Experiments logged:      {list(master_summary['experiments'].keys())}")

Master summary saved to: /content/drive/MyDrive/DP-MMFL/results/consolidated_evaluation/master_summary.json
Experiments logged:      ['E1', 'E2', 'E3', 'E4_pilot', 'E4_corrected']


In [ ]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"

df = pd.read_parquet(MANIFEST_PATH)

label_cols = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

print("Manifest shape:        ", df.shape)
print("Columns:               ", df.columns.tolist())
print("Unique patients:       ", df["deid_patient_id"].nunique())
print("Duplicate image paths: ", df["path_to_image"].duplicated().sum())

print("\n--- Label Prevalence, Uncertainty, and Missingness ---")
prevalence_records = []
for col in label_cols:
    if col not in df.columns:
        print(f"[MISSING COLUMN] {col} is absent from manifest!")
        continue

    val_counts = df[col].value_counts(dropna=False).to_dict()
    total = len(df)
    pos = val_counts.get(1.0, 0)
    neg = val_counts.get(0.0, 0)
    unc = val_counts.get(-1.0, 0)
    nan_count = df[col].isna().sum()

    prevalence_records.append({
        "Pathology": col,
        "Positive (1.0)": f"{pos} ({pos / total * 100:.2f}%)",
        "Negative (0.0)": f"{neg} ({neg / total * 100:.2f}%)",
        "Uncertain (-1.0)": f"{unc} ({unc / total * 100:.2f}%)",
        "NaN / Missing": f"{nan_count} ({nan_count / total * 100:.2f}%)",
    })

prevalence_df = pd.DataFrame(prevalence_records)
print(prevalence_df.to_string(index=False))

print("\n--- Text Column Inspection (Target Leakage Audit) ---")
text_candidate_cols = [c for c in df.columns if any(k in c.lower() for k in ["text", "report", "impression", "finding", "history", "indication"])]
print("Candidate text fields found:", text_candidate_cols)

for c in text_candidate_cols:
    non_null_count = df[c].notna().sum()
    sample_text = str(df[c].dropna().iloc[0])[:120].replace("\n", " ") if non_null_count > 0 else "N/A"
    print(f" - {c}: {non_null_count}/{len(df)} non-null | Sample: \"{sample_text}...\"")

Manifest shape:         (223462, 58)
Columns:                ['sample_id', 'path_to_image', 'path_to_dcm', 'deid_patient_id', 'report', 'age', 'sex', 'race', 'ethnicity', 'split', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'No Finding', 'target_Enlarged Cardiomediastinum', 'target_Cardiomegaly', 'target_Lung Opacity', 'target_Lung Lesion', 'target_Edema', 'target_Consolidation', 'target_Pneumonia', 'target_Atelectasis', 'target_Pneumothorax', 'target_Pleural Effusion', 'target_Pleural Other', 'target_Fracture', 'target_Support Devices', 'mask_Enlarged Cardiomediastinum', 'mask_Cardiomegaly', 'mask_Lung Opacity', 'mask_Lung Lesion', 'mask_Edema', 'mask_Consolidation', 'mask_Pneumonia', 'mask_Atelectasis', 'mask_Pneumothorax', 'mask_Pleural Effusion', 'mask_Pleural Other', 'mask_Fracture', 'mask_Support Devices', 'expe